# Aufgabenstellung

Diese Arbeit behandelt Aufgabenteil 2 des Projektes der Veranstaltung Roboterprogrammierung und beschäftigt sich mit der Erweiterung und systematischen Evaluation eines Lazy Probabilistic Roadmaps (Lazy-PRM) Verfahrens für die Pfadplanung. Der Fokus liegt auf der Entwicklung, Implementierung und Analyse verschiedener Node-Enhancementstrategien innerhalb des Lazy-PRM-Rahmens. Die Aufgabenstellung umfasst die folgenden AUfgaben:

- 2a: Entwicklung und Implementierung von mindestens drei unterschiedlichen Node-Enhancementstrategien zur gezielten Erweiterung der Roadmap.
- 2b: Anwendung, Vergleich und Animation der entwickelten Strategien anhand mehrerer Benchmark-Umgebungen für einen 2-DOF Punktroboter und einem 2DoF Planarroboter.
- 2c: Grafische Darstellung der Anzahl der Kollisonsberechnungen, Planungszeit, Roadmapgröße und Länge des Lösungspfades sowie Diskussion der Ergebnisse. 

Ziel ist es, zu untersuchen, wie unterschiedliche Node-Enhancementstrategien die Qualität, Effizienz und Robustheit von Lazy-PRM beeinflussen.


**Lazy-PRM & Enhancementstrategien**

Lazy-PRM basiert auf der klassischen Probabilistic Roadmap, unterscheidet sich jedoch durch den verzögerten Umgang mit Kollisionen.
Knoten und Kanten werden zunächst ohne aufwändige Kollisionsprüfung in die Roadmap aufgenommen. Erst wenn ein konkreter Pfad abgefragt wird, werden die benötigten Kanten schrittweise auf Kollision überprüft und bei Bedarf verworfen. Dieses „lazy“ Vorgehen reduziert initiale Rechenkosten und verschiebt Kollisionsprüfungen gezielt auf relevante Teile des Suchraums. Im Rahmen dieser Arbeit wurden fünf verschiedene Node-Enhancemethoden untersucht:

- **Baseline:** Gleichverteiltes Sampling im Konfigurationsraum
- **Mode 1:** Seed-basiertes Gauß-Sampling
- **Mode 2:** Distanz-basiertes Sampling
- **Mode 3:** Max-Min-Sampling
- **Mode 4:** Start-/Ziel-biasiertes Sampling

Jede Strategie wird im Notebook separat erläutert, implementiert und experimentell bewertet

Im folgendem Codeabschnitt werden alle benötigten Bibliotheken sowie die projektinternen Module für Benchmarking, Kollisionsprüfung und Lazy-PRM geladen. Anschließend werden die globalen Roadmapparameter und darauf aufbauend verschiedene Node-Enhancementstrategien (Baseline und Mode 1–4) definiert. Abschließend wird eine Hilfsfunktion bereitgestellt, um selbst erstellte Benchmark-Umgebungen inklusive Hindernissen sowie Start- und Zielkonfiguration zu visualisieren.

In [ ]:
# Imports
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from shapely.geometry import Polygon, LineString, Point
from IPBenchmark import Benchmark
from IPEnvironment import CollisionChecker

import IPTestSuite as ts
from IPVISLazyPRM import lazyPRMVisualize
from IPLazyPRM_Task2 import EnhancedLazyPRM

# Globale Roadmapparameter 
base_config = {
    "initialRoadmapSize": 2,    # Startgröße der Roadmap (wie viele Nodes zu Beginn)
    "updateRoadmapSize": 3,     # wie viele neue Nodes pro “Enhancement/Update”-Iteration hinzugefügt werden
    "kNearest": 4,              # wie viele Nachbarn pro Node für Kantenbildung
    "maxIterations": 90,        # maximale Iterationen (Roadmap-Erweiterungen / Lazy-PRM-Loops)

    # Robotparameter zentral in der Config
    "robotRadius": 0,        # Roboterradius
    "safetyMargin": 0,       # Sicherheitsabstand
}

# Modespezifische Konfiguration
cfg_base = dict(base_config, enhanceMode="baseline_uniform")
cfg_m1 = dict(base_config, enhanceMode="mode1_seed_gauss", seedSigma=4.0, seedTries=5)
cfg_m2 = dict(base_config, enhanceMode="mode2_seed_dist", seedMaxStep=4.0, seedBeta=0.9, seedTries=5)
cfg_m3 = dict(base_config, enhanceMode="mode3_max_min", dispersionCandidates=5)
cfg_m4 = dict(base_config, enhanceMode="mode4_start_goal_corr", corridorSigma=3.0, corridorAlongSigma=0.6, corridorTries=6)# 1.8 und 0.45

configs = {
    "baseline_uniform":       cfg_base,
    "mode1_seed_gauss":       cfg_m1,
    "mode2_seed_dist":        cfg_m2,
    "mode3_max_min":          cfg_m3,   
    "mode4_start_goal_corr":  cfg_m4,  
}

MODE_ORDER = ["baseline_uniform", "mode1_seed_gauss", "mode2_seed_dist", "mode3_max_min", "mode4_start_goal_corr"]

ROBOT_RADIUS = base_config["robotRadius"]
SAFETY = base_config["safetyMargin"]
CLEARANCE = ROBOT_RADIUS + SAFETY   # Mindestabstand zum Hindernis

def plot_benchmark(bench, bounds=(0, 22, 0, 22), ax=None,
                   obstacle_alpha=0.25, obstacle_edge="white",
                   start_color="green", goal_color="red"):
    """
    Visualisiert einen IPBenchmark.Benchmark:
    - Hindernisse aus bench.collisionChecker.scene (Shapely-Geometrien)
    - Start / Goal aus startList/goalList
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))

    xmin, xmax, ymin, ymax = bounds
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.3)
    ax.set_title(bench.name)

    # Hindernisse zeichnen
    scene = getattr(bench.collisionChecker, "scene", {})
    for _, geom in scene.items():
        if geom.is_empty:
            continue

        # Polygon / MultiPolygon
        if geom.geom_type == "Polygon":
            xs, ys = geom.exterior.xy
            ax.fill(xs, ys, alpha=obstacle_alpha, edgecolor=obstacle_edge, linewidth=2)
        elif geom.geom_type == "MultiPolygon":
            for poly in geom.geoms:
                xs, ys = poly.exterior.xy
                ax.fill(xs, ys, alpha=obstacle_alpha, edgecolor=obstacle_edge, linewidth=2)
        else:
            # Für Circles (buffer) ist es meist Polygon, aber falls nicht:
            try:
                xs, ys = geom.exterior.xy
                ax.fill(xs, ys, alpha=obstacle_alpha, edgecolor=obstacle_edge, linewidth=2)
            except Exception:
                pass

    # Start / Goal
    s = bench.startList[0]
    g = bench.goalList[0]
    ax.scatter([s[0]], [s[1]], s=120, c=start_color, label="Start", zorder=5)
    ax.scatter([g[0]], [g[1]], s=120, c=goal_color, label="Goal", zorder=5)

    ax.legend(loc="upper right")
    return ax


# Benchmarks

Für die Evaluation der Node-Enhancementstrategien sind die Benchmark-Umgebungen angepasst worden. Diese sind gezielt so gestaltet, dass sie typische Herausforderungen wie Engstellen und schmale Passagen, Bereiche mit hoher Hindernisdichte oder Strukturierte Korridore und komplexe Geometrien abbilden. Durch die kontrollierte Gestaltung der Benchmarks ist es möglich, die Vor- und Nachteile der einzelnen Node-Enhancementstrategien systematisch und reproduzierbar zu untersuchen. Insbesondere kann analysiert werden, wie sich die Strategien unter steigender Umgebungs- und Konfigurationsraumkomplexität verhalten.

In [ ]:
# Notebook Setup (einmalig)
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt

from benchmarks_task2 import (
    make_circle_field_benchmark,
    make_mode1_wall_tiny_door_closed22,
    make_mode2_wall_tiny_wide_door,
    make_mode3_u_shape_benchmark,
    make_mode4_snail,
)

## Benchmark 1: Circle Field

Bei dem Benchmark "CirckeField" sind viele runde Hindernisse gleichmäßig über den Arbeitsraum verteilt, teilweise auch randnah. Zwischen den Hindernissen existieren zahlreiche alternative Durchgänge unterschiedlicher Breite. Dieser Benchmark testet die globale Abdeckung und Homogenität der Roadmap. Erfolgreiche Strategien müssen den Raum gleichmäßig explorieren, ohne sich zu stark auf lokale Bereiche zu konzentrieren. Besonders geeignet zur Bewertung von uniformem Sampling (Baseline-Strategie) und Max-Min-Strategien (Mode 3).

In [ ]:
bench_baseline = make_circle_field_benchmark(
    clearance=CLEARANCE,
    name="CircleField",
    bounds=(0.0, 22.0, 0.0, 22.0),
    start=(2, 20),
    goal=(20, 2),
    n_circles=40,
    r_min=1.3,
    r_max=2.2,
    min_center_dist=1.0,
    inner_keepout=2.2,
    edge_overshoot=2.8,   # mehr Randüberhang
    seed=7,
)

ax = plot_benchmark(bench_baseline)
plt.show()

## Benchmark 2: WallTinyDoor

Bei dem Benchmark "WallTinyDoor" teilt ein große trennende Wand den Raum nahezu vollständig. Start und Ziel sind nur durch eine sehr schmale Öffnung miteinander verbunden. Dieser Benchmark prüft die Fähigkeit der Strategien, schmale Passagen zuverlässig zu finden und zu durchqueren. Er ist besonders sensitiv gegenüber Strategien, die lokale Exploration (Mode 1) oder gerichtetes Sampling (Mode 4 bzw. Mode 2 als Zwischenform) fördern, und stellt eine klassische Schwäche rein uniformer Verfahren dar.

In [ ]:
bench_m1 = make_mode1_wall_tiny_door_closed22(
    clearance=CLEARANCE,
    extra_block=True
)

ax = plot_benchmark(bench_m1)
ax.set_xlim(0, 22)
ax.set_ylim(0, 22)
plt.show()

## Benchmark 3: WallTinyWideDoor

Der Benchmark "WallTinyWideDoor" ist ähnlich zu "WallTinyDoor", jedoch ist die Öffnung in der Wand deutlich länger. Er besteht aus mehreren großen Hindernisstrukturen, die den freien Konfigurationsraum stark einschränken. Die Herausforderung liegt weniger in der punktuellen Auflösung eines kritischen Engpasses als vielmehr in der robusten globalen Vernetzung entlang komplexer Hindernisränder. Der Benchmark eignet sich daher besonders zur Bewertung, ob ein Planungsverfahren unnötige lokale Nachverdichtung erzeugt oder in der Lage ist, mit moderater globaler Abdeckung effizient gültige Pfade zu finden.

In [ ]:
bench_m2 = make_mode2_wall_tiny_wide_door(
    clearance=CLEARANCE,
    extra_block=True
)

ax = plot_benchmark(bench_m2)
ax.set_xlim(0, 22)
ax.set_ylim(0, 22)
plt.show()

## Benchmark 4: U-Shape

Bei Benchmark "U-Shape" umschließt ein U-förmiges Hindernis das Ziel teilweise. Der direkte Weg ist blockiert, der gültige Pfad erfordert ein gezieltes Umfahren der Struktur. Getestet wird hier die Fähigkeit zur nicht-lokalen Planung. Strategien müssen erkennen, dass ein direkter, zielgerichteter Ansatz scheitert, und stattdessen alternative, längere Wege erkunden. Besonders relevant für die Bewertung von Start–Ziel-biasierten Verfahren (Mode 4).

In [ ]:
bench_m3 = make_mode3_u_shape_benchmark(
    clearance=CLEARANCE,
    name="U-Shape",
    bounds=(0.0, 22.0, 0.0, 22.0),
)

ax = plot_benchmark(bench_m3)
ax.set_aspect("equal")
ax.set_xlim(0, 22)
ax.set_ylim(0, 22)
plt.show()

## Benchmark 5: Snail

Bei dem Benchmark "Snail" zwingt ein spiralförmiger Korridor den Roboter, einem langen, schmalen und verschlungenen Pfad zu folgen, um das Ziel im Inneren zu erreichen. Dieser Benchmark kombiniert lange schmale Passagen mit mehrfachen Richtungswechseln und stellt damit eine hohe Anforderung an Roadmap-Konnektivität und Pfadkontinuität. Er eignet sich besonders zur Analyse, ob Strategien nur lokal gute Lösungen finden oder auch komplexe, tief verschachtelte Strukturen zuverlässig erschließen.

In [ ]:
bench_m4 = make_mode4_snail(
    clearance=CLEARANCE,
    name="Snail",
    bounds=(0.0, 22.0, 0.0, 22.0),
)

ax = plot_benchmark(bench_m4)
ax.set_aspect("equal")
ax.set_xlim(0, 22)
ax.set_ylim(0, 22)
plt.show()

# Pipeline für Parametertuning

Ziel des Parametertunings ist es, für jede Node-Enhancementstrategie (Baseline sowie Mode 1–4) eine robuste Parameterkonfiguration zu bestimmen, die eine hohe Erfolgsrate bei der Pfadfindung erzielt und gleichzeitig effizient arbeitet. Effizienz wird dabei anhand mehrerer Kriterien bewertet: geringe Planungszeit, eine niedrige Anzahl an Kollisionsprüfungen sowie – nachrangig – kurze resultierende Pfade. Das Parametertuning erfolgt ausschließlich auf den selbst entwickelten Benchmarks, um eine passgenaue und aussagekräftige Bewertung der Enhancementstrategien sicherzustellen.

Optimiert werden ausschließlich die modusspezifischen Parameter der jeweiligen Node-Enhancementstrategien. Dazu zählen beispielsweise bei Mode 1 die Parameter seedSigma und seedTries, bei Mode 2 seedMaxStep, seedBeta und seedTries, bei Mode 3 die Anzahl der dispersionCandidates sowie bei Mode 4 die Korridorparameter corridorSigma, corridorAlongSigma und corridorTries. Die globalen Parameter des Lazy-PRM-Frameworks, wie etwa die initiale Roadmapgröße, die Anzahl neu hinzugefügter Nodes pro Iteration, die Anzahl der Nachbarn für die Kantenbildung sowie die maximale Anzahl an Iterationen, werden bewusst nicht optimiert. Diese bleiben für alle Modi identisch, um eine faire Vergleichbarkeit der Enhancementstrategien zu gewährleisten und deren Wirkung isoliert analysieren zu können.

Das eigentliche Parametertuning erfolgt in einem zweistufigen Verfahren. In der ersten Phase (Stage A) werden pro Mode 50 zufällige Parameterkonfigurationen aus vordefinierten Suchräumen generiert. Jede dieser Konfigurationen wird mit lediglich sechs Runs pro Benchmark evaluiert. Diese Phase dient einer breiten Exploration des Parameterraums bei vergleichsweise geringem Rechenaufwand. Ziel ist es nicht, bereits eine endgültige Entscheidung zu treffen, sondern vielversprechende Kandidaten zu identifizieren. Die Ergebnisse werden anhand einer lexikographischen Sortierung bewertet, bei der zunächst die Erfolgsrate maximiert wird. Bei gleicher Erfolgsrate werden anschließend geringere mittlere Laufzeiten, weniger Kollisionsprüfungen und kürzere Pfadlängen bevorzugt.

In der zweiten Phase (Stage B) werden die besten Kandidaten aus Stage A erneut untersucht. Pro Mode werden die acht bestplatzierten Parameterkonfigurationen ausgewählt und jeweils mit 30 Runs pro Benchmark getestet. Durch die höhere Anzahl an Wiederholungen wird der Einfluss zufälliger Effekte deutlich reduziert, sodass eine statistisch robustere Bewertung möglich ist. Auf Basis dieser Ergebnisse wird schließlich pro Mode eine einzelne Parameterkonfiguration bestimmt, die als bestgeeignete Einstellung für die anschließende Evaluation verwendet wird.

In [ ]:
"""
# ============================================================
# PARAMETER TUNING PIPELINE (Updated for your CURRENT benchmarks + modes + parameters)
# ============================================================
# Drop-in section for Projekt_Aufgabe2_LazyPRM.ipynb
#
# Assumptions (matching your current notebook setup):
# - You already have: run_suite, EnhancedLazyPRM, and your benchmark factory functions defined.
# - Current modes + parameters:
#   baseline_uniform: (no extra params)
#   mode1_seed_gauss: seedSigma, seedTries
#   mode2_seed_dist:  seedMaxStep, seedBeta, seedTries
#   mode3_max_min:    dispersionCandidates
#   mode4_start_goal_corr: corridorSigma, corridorAlongSigma, corridorTries
# - You use ONLY your own benchmarks.
# ============================================================

import math
import numpy as np
import pandas as pd


# -----------------------------
# 1) Benchmarks (robust factory lookup)
# -----------------------------
def _first_existing_factory(factory_name_candidates):
    
    #Returns a callable from globals() if any name exists, else raises a clear error.
    #This makes the tuning block resilient to different function names in your notebook.
    
    for fn_name in factory_name_candidates:
        fn = globals().get(fn_name, None)
        if callable(fn):
            return fn
    raise NameError(
        "No benchmark factory found. Tried: "
        + ", ".join(factory_name_candidates)
        + "\nPlease adjust the candidate names to match your notebook."
    )


def build_all_benchmarks():
    benchmarks = [
        ("CircleField",      make_circle_field_benchmark()),
        ("TinyDoorClosed22", make_mode1_wall_tiny_door_closed22()),
        ("TinyWideDoor",     make_mode2_wall_tiny_wide_door()),
        ("U-Shape",          make_mode3_s_corridor_benchmark()),  # <- das IST dein U-Shape
        ("Snail",            make_mode4_snail()),
    ]

    out = []
    for bname, b in benchmarks:
        if hasattr(b, "name"):
            b.name = bname
        out.append(b)

    print("[build_all_benchmarks] loaded:", [b.name for b in out])
    return out



# -----------------------------
# 2) Sampling helpers
# -----------------------------
def _rand_int(rng, lo, hi):
    return int(rng.integers(lo, hi + 1))

def _rand_float(rng, lo, hi):
    return float(rng.uniform(lo, hi))

def _rand_loguniform(rng, lo, hi):
    lo = float(lo); hi = float(hi)
    if lo <= 0 or hi <= 0:
        raise ValueError("loguniform bounds must be > 0")
    return float(math.exp(rng.uniform(math.log(lo), math.log(hi))))


# -----------------------------
# 3) Search spaces
# -----------------------------
def global_space():
    
    #Global roadmap parameters. Keep these ranges moderate; your current setup
    #already uses small-ish roadmaps to make enhancement differences visible.
    
    return {
        "initialRoadmapSize": lambda rng: _rand_int(rng, 2, 12),
        "updateRoadmapSize":  lambda rng: _rand_int(rng, 2, 10),
        "kNearest":           lambda rng: _rand_int(rng, 2, 12),
        "maxIterations":      lambda rng: _rand_int(rng, 30, 120),
    }


def param_space_for_mode(mode_name):
    if mode_name == "baseline_uniform":
        return {}

    if mode_name == "mode1_seed_gauss":
        return {
            # Larger sigma => broader local exploration around seed nodes
            "seedSigma": lambda rng: _rand_loguniform(rng, 0.2, 8.0),
            # How many attempts to get a valid sample
            "seedTries": lambda rng: _rand_int(rng, 2, 20),
        }

    if mode_name == "mode2_seed_dist":
        return {
            # Max step size from a seed/anchor
            "seedMaxStep": lambda rng: _rand_loguniform(rng, 0.3, 8.0),
            # Bias/weighting parameter (clipped to [0,1])
            "seedBeta":    lambda rng: _rand_float(rng, 0.2, 1.0),
            # Attempts to obtain valid samples
            "seedTries":   lambda rng: _rand_int(rng, 2, 20),
        }

    if mode_name == "mode3_max_min":
        return {
            # Number of candidate samples for Max-Min selection per iteration
            "dispersionCandidates": lambda rng: _rand_int(rng, 5, 80),
        }

    if mode_name == "mode4_start_goal_corr":
        return {
            # Orthogonal corridor width around the start-goal line
            "corridorSigma":      lambda rng: _rand_loguniform(rng, 0.2, 6.0),
            # Spread along the start-goal direction
            "corridorAlongSigma": lambda rng: _rand_loguniform(rng, 0.05, 2.0),
            # Attempts to get a valid corridor sample
            "corridorTries":      lambda rng: _rand_int(rng, 2, 30),
        }

    raise ValueError(f"Unknown mode: {mode_name}")


def sample_config(base_config, mode_name, rng, tune_globals=True):
    
    #Creates one candidate configuration:
    #- copies base_config
    #- sets enhanceMode
    #- optionally tunes global roadmap parameters
    #- tunes the mode-specific parameters
    #- keeps robotRadius/safetyMargin untouched unless you explicitly include them in base_config changes
    
    cfg = dict(base_config)
    cfg["enhanceMode"] = mode_name

    if tune_globals:
        for k, sampler in global_space().items():
            cfg[k] = sampler(rng)

    for k, sampler in param_space_for_mode(mode_name).items():
        cfg[k] = sampler(rng)

    # Plausibility rules / clipping
    if mode_name == "mode2_seed_dist":
        cfg["seedBeta"] = float(np.clip(cfg["seedBeta"], 0.0, 1.0))
        cfg["seedTries"] = int(max(1, cfg["seedTries"]))
        cfg["seedMaxStep"] = float(max(1e-6, cfg["seedMaxStep"]))

    if mode_name == "mode1_seed_gauss":
        cfg["seedSigma"] = float(max(1e-6, cfg["seedSigma"]))
        cfg["seedTries"] = int(max(1, cfg["seedTries"]))

    if mode_name == "mode3_max_min":
        cfg["dispersionCandidates"] = int(max(1, cfg["dispersionCandidates"]))

    if mode_name == "mode4_start_goal_corr":
        cfg["corridorSigma"] = float(max(1e-6, cfg["corridorSigma"]))
        cfg["corridorAlongSigma"] = float(max(1e-6, cfg["corridorAlongSigma"]))
        cfg["corridorTries"] = int(max(1, cfg["corridorTries"]))

    # ensure ints for these globals if tuned
    for k in ("initialRoadmapSize", "updateRoadmapSize", "kNearest", "maxIterations"):
        if k in cfg:
            cfg[k] = int(cfg[k])

    return cfg


# -----------------------------
# 4) Aggregation + ranking
# -----------------------------
def summarize_df(df):
    
    #Expects df columns like run_suite:
    #benchmark, mode, run, success, time_s, collision_checks, path_length, roadmap_size
    
    df2 = df.copy()
    metric_cols = ["time_s", "collision_checks", "path_length", "roadmap_size"]
    for c in metric_cols:
        df2.loc[df2["success"] == False, c] = np.nan

    g = df2.groupby(["candidate_id"]).agg(
        runs=("run", "count"),
        success_rate=("success", "mean"),
        time_mean=("time_s", "mean"),
        time_std=("time_s", "std"),
        collision_checks_mean=("collision_checks", "mean"),
        path_length_mean=("path_length", "mean"),
        roadmap_size_mean=("roadmap_size", "mean"),
    ).reset_index()

    return g


def pick_best(summary_df):
    
    #Lexicographic ranking:
      #1) success_rate max
      #2) time_mean min
      #3) collision_checks_mean min
      #4) path_length_mean min
    
    s = summary_df.copy()
    s["time_mean"] = s["time_mean"].fillna(np.inf)
    s["collision_checks_mean"] = s["collision_checks_mean"].fillna(np.inf)
    s["path_length_mean"] = s["path_length_mean"].fillna(np.inf)

    s = s.sort_values(
        by=["success_rate", "time_mean", "collision_checks_mean", "path_length_mean"],
        ascending=[False, True, True, True],
        kind="mergesort"
    )
    return s.iloc[0]


# -----------------------------
# 5) Stage A / Stage B tuning
# -----------------------------
def tune_mode(
    planner_class,
    benchmarks,
    base_config,
    mode_name,
    n_candidates_stageA=40,
    runs_stageA=6,
    top_k_stageB=6,
    runs_stageB=30,
    base_seed=2025,
    tune_globals=True,
    progress_every=50,
    verbose=True,
):
    rng = np.random.default_rng(base_seed + abs(hash(mode_name)) % 10_000)

    candidate_cfgs = {}
    all_rows_A = []

    if verbose:
        print(f"\n[Stage A] mode={mode_name} | candidates={n_candidates_stageA} | runs/cand={runs_stageA}")

    for cid in range(n_candidates_stageA):
        cfg = sample_config(base_config, mode_name, rng, tune_globals=tune_globals)
        candidate_cfgs[cid] = cfg

        bench_mode_map = {b.name: [mode_name] for b in benchmarks}
        configs = {mode_name: cfg}

        df = run_suite(
            benchmarks=benchmarks,
            configs=configs,
            runs=runs_stageA,
            base_seed=int(base_seed) + 10_000 * int(cid),
            progress_every=progress_every,
            bench_mode_map=bench_mode_map,
        )
        df["candidate_id"] = cid
        all_rows_A.append(df)

    dfA = pd.concat(all_rows_A, ignore_index=True)

    rankedA = summarize_df(dfA).sort_values(
        by=["success_rate", "time_mean", "collision_checks_mean", "path_length_mean"],
        ascending=[False, True, True, True],
        kind="mergesort"
    ).reset_index(drop=True)

    top_ids = list(rankedA["candidate_id"].head(top_k_stageB).values)

    if verbose:
        print(f"[Stage A] Top-{top_k_stageB}: {top_ids}")
        print(rankedA.head(min(10, len(rankedA))))

    # ---- Stage B
    all_rows_B = []
    if verbose:
        print(f"\n[Stage B] mode={mode_name} | confirm={len(top_ids)} | runs/cand={runs_stageB}")

    for cid in top_ids:
        cfg = candidate_cfgs[int(cid)]

        bench_mode_map = {b.name: [mode_name] for b in benchmarks}
        configs = {mode_name: cfg}

        df = run_suite(
            benchmarks=benchmarks,
            configs=configs,
            runs=runs_stageB,
            base_seed=int(base_seed) + 99_000 * int(cid),
            progress_every=progress_every,
            bench_mode_map=bench_mode_map,
        )
        df["candidate_id"] = int(cid)
        all_rows_B.append(df)

    dfB = pd.concat(all_rows_B, ignore_index=True)

    rankedB = summarize_df(dfB).sort_values(
        by=["success_rate", "time_mean", "collision_checks_mean", "path_length_mean"],
        ascending=[False, True, True, True],
        kind="mergesort"
    ).reset_index(drop=True)

    best_row = pick_best(rankedB)
    best_id = int(best_row["candidate_id"])
    best_cfg = candidate_cfgs[best_id]

    if verbose:
        print(f"\n[BEST] mode={mode_name} | best_candidate_id={best_id}")
        print("Best summary:", best_row.to_dict())
        print("Best config:", best_cfg)

    return {
        "mode": mode_name,
        "best_candidate_id": best_id,
        "best_config": best_cfg,
        "summary_stageA": rankedA,
        "summary_stageB": rankedB,
        "raw_stageA": dfA,
        "raw_stageB": dfB,
    }


def tune_all_modes(
    planner_class,
    base_config,
    base_seed=2025,
    tune_globals=True,
    n_candidates_stageA=50,
    runs_stageA=6,
    top_k_stageB=8,
    runs_stageB=30,
    progress_every=50,
    verbose=True,
):
    benchmarks = build_all_benchmarks()

    modes = [
        "baseline_uniform",
        "mode1_seed_gauss",
        "mode2_seed_dist",
        "mode3_max_min",
        "mode4_start_goal_corr",
    ]

    tuned = {}
    for m in modes:
        tuned[m] = tune_mode(
            planner_class=planner_class,
            benchmarks=benchmarks,
            base_config=base_config,
            mode_name=m,
            n_candidates_stageA=n_candidates_stageA,
            runs_stageA=runs_stageA,
            top_k_stageB=top_k_stageB,
            runs_stageB=runs_stageB,
            base_seed=base_seed,
            tune_globals=tune_globals,
            progress_every=progress_every,
            verbose=verbose,
        )

    best_configs = {m: tuned[m]["best_config"] for m in modes}
    return tuned, best_configs


# ============================================================
# RUN: Tune all modes (example)
# ============================================================
# Keep your robot parameters here too, so tuning uses the same collision model.
base_config = {
    "initialRoadmapSize": 2,
    "updateRoadmapSize":  3,
    "kNearest":           4,
    "maxIterations":      90,

    "robotRadius": 0.25,
    "safetyMargin": 0.05,
}

tuned, best_configs = tune_all_modes(
    planner_class=EnhancedLazyPRM,
    base_config=base_config,
    base_seed=2025,

    # Budget (adjust to runtime)
    n_candidates_stageA=50,
    runs_stageA=6,
    top_k_stageB=8,
    runs_stageB=30,

    # As in your notebook rationale: keep globals fixed if you want baseline weak + enhancements visible
    tune_globals=False,

    progress_every=50,
    verbose=True,
)

print("\nBest configs per mode:")
for mode, cfg in best_configs.items():
    print(f"\n--- {mode} ---")
    for k, v in cfg.items():
        print(f"{k}: {v}")
"""

# Darstellung und Evaluation der Node-Enhancementstrategien

In diesem Kapitel werden die implementierten Node-Enhancementstrategien vorgestellt und systematisch miteinander verglichen. Ziel ist es, die Auswirkungen der unterschiedlichen Sampling- und Erweiterungsmechanismen auf die Struktur der Roadmap sowie auf die Qualität und Effizienz der resultierenden Pfade zu analysieren. Als Referenz dient eine Baseline mit uniformem Sampling, gegenüber der gezielte, lokal- oder zielgerichtete Erweiterungsstrategien evaluiert werden. Die Darstellung erfolgt sowohl qualitativ anhand der erzeugten Roadmaps und Pfade als auch quantitativ mittels aggregierter Metriken aus den Benchmark-Experimenten. In diesem Abschnitt wird die Robotergeometrie nicht berücksichtig, erst in den späteren Abschnitten zu 2DoF-Punkt- & Planarroboter.

Allen Varianten gemeinsam ist der Lazy-PRM-Grundgedanke: Kanten werden nicht sofort vollständig kollisionsgeprüft, sondern erst dann validiert, wenn ein konkreter Pfad zwischen Start und Ziel gesucht wird. Kanten, die dabei als kollidierend erkannt werden, sind aus der Roadmap zu entfernen. Unabhängig vom Modus wird zu Beginn eine initiale Roadmap erzeugt. Hier wird die initiale Roadmap mit nur zwei Knoten bewusst minimal gewählt, um den Effekt der Node-Enhancementstrategien beim schrittweisen Ausbau der Roadmap deutlich hervorzuheben. Diese initialen Knoten werden zufällig im Konfigurationsraum gesampelt; ungültige Samples welche sich außerhalb der Roadmap befinden werden verworfen, bis genügend gültige Startknoten vorliegen. Die Verknüpfung dieser Knoten erfolgt über kNearest welche gemäß Lazy-PRM erst bei Bedarf validiert werden.

**Mode 0 Baseline**

Der Mode Baseline erweitert die Roadmap während des gesamten Planungsprozesses durch uniformes, ungerichtetes Sampling. Neue Knoten werden zufällig im Konfigurationsraum erzeugt und in die Roadmap integriert; eine gezielte Nachbesserung an Stellen, an denen zuvor Kanten aufgrund von Kollisionen entfernt wurden, findet nicht statt. Dadurch dient die Baseline als Referenz, um den Nutzen gezielter Enhancementstrategien isoliert beurteilen zu können.

**Mode 1 Seed Gauss**

Mode 1 setzt auf Verdichtung an den tatsächlich kritischen Stellen, die durch den Lazy-Mechanismus sichtbar werden. Sobald beim Pfadversuch eine Kante als kollidierend erkannt und gelöscht wird, wird aus ihrem Mittelpunkt ein Seedpoint gebildet. Um diese Seedpoints werden anschließend neue Knoten mittels Gauß-Sampling erzeugt, wodurch die Roadmap genau dort dichter wird, wo zuvor eine Verbindung fehlgeschlagen ist. Gauß-Sampling bedeutet das Nodes mit hoher Dichte nahe am Seed und abnehmender Dichte mit wachsender Entfernung gesampelt werden. Die Parameter steuern dabei im Kern, wie breit um den Seed gestreut wird und wie viele Versuche pro Seed unternommen werden, um gültige Samples zu erhalten. Mit dieser Strategie soll durch lokale zusätzliche Knoten, alternative kollisionsfreie Verbindungen in der Umgebung einer fehlgeschlagenen Kante ermöglicht werden. Parameter:

- seedSigma: Standardabweichung der Gauß-Verteilung um einen Seed-Knoten; größer = breiteres, explorativeres Sampling, kleiner = lokaler.
- seedTries: Anzahl Nodes welche pro Seed gesampelt werden

**Mode 2 Seed Distance**

Mode 2 verwendet ebenfalls Seedpoints aus den Mittelpunkten gelöschter Kanten, unterscheidet sich jedoch in der Art, wie neue Knoten um diese Seeds erzeugt werden. Anstatt gaußveteilt um den Seed zu verdichten, werden neue Knoten in kontrollierter Distanz zum Seed erzeugt. Für jeden neuen Knoten wird vom Seed aus eine zufällige Richtung im Konfigurationsraum gewählt. Mode 2 kann als lokalisierte Variante des Baseline-Samplings verstanden werden. Anstatt global im gesamten Konfigurationsraum zu sampeln, werden mehrere zufällige Samples in einem begrenzten Umfeld eines Seeds erzeugt. Dadurch bleibt die Samplingstrategie einfach, wird jedoch auf problemrelevante Regionen fokussiert. Parameter:

- seedMaxStep: Maximale Schrittweite bzw. maximale Verschiebung von dem Seedpoint.
- seedBeta: Skaliert/limitiert die tatsächliche Schrittweite
- seedTries: Anzahl Nodes welche pro Seed gesampelt werden

**Mode 3 Max-Min Sampling**

Mode 3 verfolgt ein explizit abdeckungsorientiertes Prinzip und nutzt ein Max-Min-Kriterium zur Auswahl neuer Knoten. Pro Erweiterungsschritt werden zunächst mehrere Kandidaten zufällig (typischerweise uniform) im Konfigurationsraum gesampelt. Für jeden Kandidaten wird der Abstand zum nächstgelegenen existierenden Roadmap-Knoten bestimmt. Aus allen Kandidaten wird dann der ausgewählt, dessen minimaler Abstand am größten ist. Dieses Vorgehen erhöht systematisch die Dispersion der Roadmap und verbessert die globale Abdeckung, wodurch alternative, bislang nicht erschlossene Umgehungsrouten wahrscheinlicher werden – insbesondere dann, wenn die bisherige Roadmap aufgrund gelöschter Kanten in bestimmten Bereichen keine tragfähige Verbindung herstellen konnte. Parameter:

- dispersionCandidates: Anzahl zufällig erzeugter Kandidaten pro Iteration.

**Mode 4 Start-Goal Corridor**

Mode 4 ist eine zielgerichtete Strategie, die das Sampling auf pfadrelevante Regionen fokussiert. Anstatt neue Knoten im gesamten Konfigurationsraum zu erzeugen, werden Kandidaten bevorzugt in einem Korridor entlang der Verbindung zwischen Start- und Zielkonfiguration gesampelt. Dazu wird typischerweise eine Position entlang der Start–Ziel-Richtung gewählt und anschließend eine orthogonale Abweichung hinzugefügt, sodass sich ein bandförmiger Sampling-Bereich um die Verbindungslinie ergibt. Die Streuung quer zur Achse bestimmt die Breite des Korridors, während die Streuung entlang der Achse die Verteilung der Samples in Richtung Start–Ziel beeinflusst; ungültige Kandidaten werden verworfen und bis zu einer festgelegten Anzahl von Versuchen neu erzeugt. In Kombination mit Lazy-PRM führt dieser Ansatz dazu, dass Kollisionsprüfungen weiterhin nur auf tatsächlich relevanten Kanten erfolgen, während die Roadmap-Erweiterung räumlich dort konzentriert wird, wo eine Start–Ziel-Verbindung am wahrscheinlichsten entsteht.

- corridorSigma: Querstreuung des Korridors um die Start–Ziel-Achse; größer = breiterer Korridor, kleiner = stärker fokussiert.
- corridorAlongSigma: Streuung entlang der Start–Ziel-Richtung; steuert, wie konzentriert entlang der Achse gesampelt wird.
- corridorTries: Anzahl Versuche, einen gültigen Sample innerhalb des Korridors zu finden.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt

from eval_task2 import (
    run_suite,
    summarize,
    plot_benchmark_like_task1,
    extract_best_seed_map,
    visualize_best_per_mode_and_benchmark,
)

## Evaluation mit Benchmark 1: Baseline_CircleField

Für den Benchmark CircleField wurden alle fünf Sampling-Modi (Baseline sowie Mode 1–4) jeweils mit 30 unabhängigen Planungsdurchläufen evaluiert. Das grüne Balkendiagramm gibt an, wie viele Runs erfolgreich waren und zu einem gültigen Pfad geführt haben. Die übrigen Balkendiagramme zeigen jeweils den Mittelwert und die Varianz der betrachteten Metriken über alle erfolgreichen Läufe. Die darunter dargestellten Roadmap- und Pfadvisualisierungen repräsentieren hingegen einen Best-Case pro Modus und dienen der qualitativen Veranschaulichung typischer Lösungsstrukturen.
Die Auswahl dieses Best-Cases erfolgt nicht anhand einer einzelnen Metrik, sondern über eine gewichtete Mehrkriterienbewertung. Dazu werden zunächst nur erfolgreiche Läufe berücksichtigt und extreme Ausreißer verworfen, indem die betrachteten Metriken auf ein zentrales Quantilintervall (z. B. 5 % bis 95 %) beschränkt werden. Anschließend wird für jeden verbleibenden Lauf ein kombinierter Score berechnet, der sich aus einer gewichteten Summe von Planungszeit, Pfadlänge und Roadmapgröße zusammensetzt. Die Gewichtung priorisiert dabei die Planungszeit am stärksten, gefolgt von der Pfadlänge und der Größe der Roadmap. Der Lauf mit dem minimalen Gesamtscore wird als Best-Case ausgewählt und grafisch dargestellt.
Wichtig ist, dass dieser Best-Case nicht repräsentativ für den Mittelwert der jeweiligen Metriken ist, sondern einen besonders günstigen Einzelverlauf zeigt. 

In [ ]:
benchmarks = [bench_baseline]

df = run_suite(
    planner_class=EnhancedLazyPRM,
    benchmarks=benchmarks,
    configs=configs,
    runs=30,
    base_seed=2025,
    progress_every=10,
    bench_mode_map=None
)

summary = summarize(df)

# 1) Summary-Plots (1×4)
for b in benchmarks:
    plot_benchmark_like_task1(summary, b.name, mode_order=MODE_ORDER)

# 2) Seeds aus denselben Evaluation-Ergebnissen ziehen (optional)
SEED_MAP = extract_best_seed_map(df, prefer_seed_runs_for_mode12=True)

# 3) Best-Run Visualisierung (Sampling-Struktur sichtbar)
BENCH_ORDER = [b.name for b in benchmarks]
visualize_best_per_mode_and_benchmark(
    df=df,
    benchmarks=benchmarks,
    configs=configs,
    planner_class=EnhancedLazyPRM,
    lazyPRMVisualize=lazyPRMVisualize,
    nodeSize=80,
    mode_order=MODE_ORDER,
    bench_order=BENCH_ORDER,
    figsize=(7, 7),
    prefer_seed_runs_for_mode12=True,
)

### Diskussion der Ergebnisse

**Planungszeit**

Die Planungszeit zeigt im CircleField moderate, aber klar erkennbare Unterschiede zwischen den Sampling-Modi. Die Varianzen überlappen teilweise, dennoch lassen sich stabile Rangfolgen erkennen. Die Baseline weist die höchste mittlere Planungszeit auf und zeigt zudem eine vergleichsweise große Streuung. Das uniforme Sampling erzeugt viele Kandidaten in wenig relevanten Bereichen, was zu einem erhöhten Aufwand durch Kollisions- und Verbindungsprüfungen führt. Mode 1 (seed_gauss) reduziert die Planungszeit gegenüber der Baseline deutlich. Durch die lokale Erweiterung um Seeds wird der effektive Suchraum eingeschränkt, ohne nennenswerten zusätzlichen algorithmischen Aufwand zu verursachen. Dadurch ergibt sich eine effiziente Planung mit geringer mittlerer Laufzeit. Mode 2 (seed_dist) weist eine etwas höhere mittlere Planungszeit als Mode 1 auf. Die distanzbasierte Seed-Strategie erzeugt zwar strukturierte Erweiterungen, erfordert jedoch zusätzliche Prüfungen zur Einhaltung der Distanzkriterien sowie eine hohe lokale Knotendichte, was den Laufzeitaufwand erhöht. Mode 3 (Max–Min) liegt hinsichtlich der Planungszeit oberhalb Mode 1 und Mode 2. Der zusätzliche Aufwand zur Bewertung mehrerer Kandidaten pro Iteration führt zu einer erhöhten Planungszeit als bei stärker lokal verdichtenden Strategien. Mode 4 (start–goal corridor) erreicht die niedrigste mittlere Planungszeit. Die explizite Einschränkung des Samplings auf den Start–Ziel-Korridor reduziert den effektiven Suchraum deutlich und vermeidet großflächige Exploration. Dadurch ergibt sich im Mittel der geringste Planungsaufwand.

**Kollisionschecks**

Die Anzahl der Kollisionsprüfungen unterscheidet sich zwischen den Sampling-Modi deutlich und erlaubt eine klare Einordnung der jeweiligen Effizienz. Die Baseline verursacht die höchste mittlere Anzahl an Kollisionschecks und weist zudem die größte Varianz auf. Das uniforme Sampling prüft den gesamten Konfigurationsraum gleichmäßig und erzeugt dadurch viele Knoten- und Kantenkandidaten in hindernisnahen oder strukturell irrelevanten Bereichen. Dies führt zu einem hohen Anteil verworfener Prüfungen. Mode 1 (seed_gauss) reduziert die Anzahl der Kollisionschecks gegenüber der Baseline deutlich. Durch die lokale Erweiterung um Seeds werden bevorzugt bereits teilweise valide Regionen nachverdichtet. Dadurch sinkt die Wahrscheinlichkeit für ungültige Kandidaten, auch wenn weiterhin explorative Anteile enthalten sind. Mode 2 (seed_dist) erreicht die geringste mittlere Anzahl an Kollisionschecks. Die distanzbasierte Seed-Strategie erzeugt neue Knoten in kontrollierter Entfernung zu bestehenden Strukturen und vermeidet sowohl zu nahe als auch zu weit entfernte Kandidaten. Dies minimiert kollisionsanfällige Prüfungen besonders effektiv. Mode 3 (Max–Min) liegt hinsichtlich der Kollisionschecks über Mode 1 und Mode 2, aber klar unterhalb der Baseline. Die gezielte Exploration schlecht abgedeckter Regionen führt häufiger zu hindernisnahen Kandidaten als bei den Seed-basierten Verfahren. Gleichzeitig begrenzt die geringe Anzahl neu eingefügter Knoten den Gesamtaufwand, sodass die Kollisionschecks moderat bleiben. Mode 4 (start–goal corridor) liegt leicht über Mode 2 und in etwa auf dem Niveau von Mode 1. Die Einschränkung des Samplings auf den Start–Ziel-Korridor reduziert zwar die globale Exploration, dieser Korridor schneidet im CircleField jedoch mehrfach Hindernisse. Dadurch bleiben kollisionsanfällige Prüfungen relevant, ohne das hohe Niveau der Baseline zu erreichen.

**Pfadlänge**

Die Pfadlängen unterscheiden sich im CircleField moderat, zeigen jedoch eine klare Rangfolge der Sampling-Modi. Die Varianzen überlappen teilweise, dennoch sind die relativen Trends stabil. Die Baseline weist eine vergleichsweise hohe mittlere Pfadlänge auf. Das uniforme Sampling erzeugt eine ungerichtete Roadmap, in der gültige Pfade häufig indirekt verlaufen und mehrere Richtungswechsel enthalten. Ohne gezielte Strukturierung entlang günstiger Routen entstehen längere Umwege. Mode 1 (seed_gauss) reduziert die Pfadlänge gegenüber der Baseline deutlich. Die lokale Verdichtung um Seeds verbessert die Konnektivität entlang bereits vielversprechender Bereiche, wodurch direktere Verbindungen entstehen können. Die Verbesserung ist konsistent, aber begrenzt. Mode 2 (seed_dist) erzielt die kürzeste mittlere Pfadlänge unter allen Modi. Die kontrollierte distanzbasierte Erweiterung begünstigt eine gleichmäßige, gut vernetzte Struktur entlang des Pfades und erlaubt effizientere Abkürzungen zwischen relevanten Knoten. Mode 3 (Max–Min) weist die größte mittlere Pfadlänge auf und zeigt zudem eine erhöhte Varianz. Die geringe Knotendichte und die globale Verteilung der Knoten führen zu gröberen Pfadverläufen mit längeren Kanten und weniger lokalen Optimierungsmöglichkeiten. Trotz guter Abdeckung des Konfigurationsraums sind die resultierenden Pfade weniger direkt. Mode 4 (start–goal corridor) erreicht eine Pfadlänge auf dem Niveau von Mode 1, jedoch leicht über Mode 2. Die Start–Ziel-Fokussierung begünstigt grundsätzlich direkte Verbindungen, im CircleField wird dieser Vorteil jedoch durch die Hindernisanordnung eingeschränkt, sodass der Korridor mehrfach korrigiert werden muss.

**Roadmapgröße**

Die Roadmapgröße zeigt im CircleField die deutlichsten Unterschiede zwischen den Sampling-Modi und erlaubt eine klare Einordnung der strukturellen Eigenschaften der jeweiligen Strategien. Die Baseline erzeugt Roadmaps mittlerer Größe mit hoher Varianz. Das uniforme Sampling führt weder zu einer gezielten Ausdünnung noch zu einer systematischen Verdichtung, sodass die Anzahl der erzeugten Knoten stark vom zufälligen Verlauf der Exploration abhängt. Mode 1 (seed_gauss) erzeugt im Mittel etwas kleinere Roadmaps als die Baseline. Die lokale Erweiterung um Seeds fokussiert die Knotenerzeugung auf bereits relevante Regionen und reduziert damit unnötige globale Exploration. Die Varianz bleibt jedoch vergleichsweise hoch, da die Effektivität der Seeds stark vom jeweiligen Run abhängt. Mode 2 (seed_dist) weist die größten Roadmaps unter allen Modi auf und zeigt zugleich eine sehr hohe Streuung. Die distanzbasierte Seed-Strategie erzeugt eine starke lokale Nachverdichtung entlang bestehender Strukturen, um Konnektivität und Pfadqualität zu maximieren. Dies führt zu einer hohen Knotenzahl, ohne dass die Exploration frühzeitig begrenzt wird. Mode 3 (Max–Min) erzeugt mit Abstand die kleinsten Roadmaps und weist zudem die geringste Varianz auf. Pro Iteration wird ausschließlich ein Knoten mit maximalem Mindestabstand zur bestehenden Roadmap ergänzt. Dies führt zu einer sehr sparsamen, gleichmäßig verteilten Abdeckung des Konfigurationsraums und einer hohen strukturellen Effizienz. Mode 4 (start–goal corridor) liegt hinsichtlich der Roadmapgröße zwischen Mode 1 und Mode 3. Die Einschränkung des Samplings auf den Start–Ziel-Korridor reduziert die globale Exploration deutlich, erfordert jedoch weiterhin zusätzliche Knoten zur Umgehung von Hindernissen entlang des Korridors. Dadurch ergibt sich eine moderate Knotenzahl mit mittlerer Varianz.

**Gesamtbewertung und Eignung für den Benchmark**

Der CircleField-Benchmark ist durch einen offenen Konfigurationsraum mit vielen gleichwertigen Durchgängen geprägt und stellt keine hohen Anforderungen an gezielte Start–Ziel-Führung oder präzise lokale Verdichtung. Entsprechend erreichen nahezu alle Modi eine hohe Erfolgsrate (überwiegend 27–30 erfolgreiche Runs), wobei Unterschiede weniger durch das Finden eines Pfades als durch Effizienz und Strukturqualität bestimmt werden. Strategien mit moderater Strukturierung oder globaler Abdeckung liefern ähnliche Pfadqualitäten, während stark gerichtete oder stark lokal verdichtende Ansätze keinen entscheidenden Vorteil erzielen. Insgesamt fungiert CircleField als Robustheits- und Effizienzbenchmark, in dem unterschiedliche Samplingstrategien zuverlässig funktionieren und sich primär in Aufwand, Roadmapgröße und Pfadqualität unterscheiden, nicht jedoch in der Lösbarkeit.

## Evaluation mit Benchmark 2: WallTinyDoor

Der Benchmark WallTinyDoor stellt eine klassische Engstellen-Situation dar: Start und Ziel sind durch eine nahezu geschlossene Wand getrennt, die nur eine sehr schmale Passage erlaubt. Alle fünf Sampling-Modi wurden jeweils mit 30 unabhängigen Planungsdurchläufen evaluiert. Die Balkendiagramme zeigen erneut Mittelwert und Varianz der betrachteten Metriken über alle erfolgreichen Läufe, während die darunter dargestellten Roadmap- und Pfadvisualisierungen jeweils einen Best-Case repräsentieren. Dieser Best-Case wird über eine gewichtete Mehrkriterienbewertung aus Planungszeit, Pfadlänge und Roadmapgröße bestimmt und dient der qualitativen Veranschaulichung typischer Lösungsstrukturen, nicht der Darstellung eines Mittelwerts.

In [ ]:
benchmarks = [bench_m1]

df = run_suite(
    planner_class=EnhancedLazyPRM,
    benchmarks=benchmarks,
    configs=configs,
    runs=30,
    base_seed=2025,
    progress_every=10,
    bench_mode_map=None
)

summary = summarize(df)

for b in benchmarks:
    plot_benchmark_like_task1(summary, b.name, mode_order=MODE_ORDER)

SEED_MAP = extract_best_seed_map(df, prefer_seed_runs_for_mode12=True)

BENCH_ORDER = [b.name for b in benchmarks]
visualize_best_per_mode_and_benchmark(
    df=df,
    benchmarks=benchmarks,
    configs=configs,
    planner_class=EnhancedLazyPRM,
    lazyPRMVisualize=lazyPRMVisualize,
    nodeSize=80,
    mode_order=MODE_ORDER,
    bench_order=BENCH_ORDER,
    figsize=(7, 7),
    prefer_seed_runs_for_mode12=True,
)

### Diskussion der Ergebnisse

**Planungszeit**

Die Planungszeit zeigt im WallTinyDoor-Benchmark deutlich stärkere Unterschiede zwischen den Sampling-Modi als im CircleField. Ursache ist der schmale Engpass, der eine präzisere Platzierung von Knoten und eine gezielte Konnektivität erfordert. Die Baseline weist eine sehr geringe mittlere Planungszeit auf. Das uniforme Sampling erzeugt schnell eine Roadmap, profitiert hier jedoch davon, dass bereits wenige zufällig platzierte Knoten ausreichen können, um den Engpass zu treffen. Die geringe Laufzeit geht allerdings mit einer geringeren strukturellen Zuverlässigkeit einher, was sich nicht in der Planungszeit selbst widerspiegelt. Mode 1 (seed_gauss) zeigt eine deutlich erhöhte Planungszeit gegenüber der Baseline. Die lokale Verdichtung um Seeds führt im engen Durchgang zu vielen zusätzlichen Kollisions- und Verbindungsprüfungen. Da Seeds häufig nahe am Hindernisrand liegen, steigt der Aufwand pro Iteration spürbar. Mode 2 (seed_dist) weist ebenfalls eine höhere mittlere Planungszeit auf. Die distanzbasierte Seed-Strategie erzeugt eine sehr hohe Knotendichte im Bereich des Engpasses, um Konnektivität sicherzustellen. Diese starke lokale Nachverdichtung führt zu zahlreichen zusätzlichen Prüfungen und damit zum größten Laufzeitaufwand. Mode 3 (Max–Min) eine höhere mittlere Planungszeit und die größte Varianz auf. Die globale Explorationsstrategie ergänzt vergleichsweise wenige Knoten, vermeidet jedoch ineffiziente lokale Überverdichtung im Engpass. Der zusätzliche Planungsaufwand durch die Kandidatenbewertung führt erneut zu einer erhöhten mittleren Planungszeit. Mode 4 (start–goal corridor) erreicht eine Planungszeit auf dem Niveau der Baseline und ist einer der schnellsten Modi. Die gezielte Einschränkung des Samplings auf den Start–Ziel-Korridor fokussiert die Exploration direkt auf den relevanten Engpass und vermeidet unnötige Prüfungen in irrelevanten Bereichen.

**Kollisionschecks**

Die Anzahl der Kollisionsprüfungen zeigt im WallTinyDoor-Benchmark klare Unterschiede zwischen den Sampling-Modi. Der schmale Engpass verstärkt die Effekte lokaler Verdichtung und gezielter Exploration deutlich stärker als in offenen Szenarien. Die Baseline weist die geringste mittlere Anzahl an Kollisionschecks auf. Das uniforme Sampling erzeugt vergleichsweise wenige Knoten und Kanten, wodurch auch die Anzahl der Prüfungen niedrig bleibt. Diese Effizienz ist jedoch rein quantitativ zu verstehen und geht nicht zwangsläufig mit einer zuverlässigen Erschließung des Engpasses einher. Mode 1 (seed_gauss) verursacht deutlich mehr Kollisionschecks als die Baseline. Die lokale Verdichtung um Seeds führt dazu, dass viele Kandidaten in unmittelbarer Nähe der Engpasswände entstehen, was die Wahrscheinlichkeit von Kollisionen erhöht. Entsprechend steigt der Anteil verworfener Prüfungen spürbar. Mode 2 (seed_dist) liegt auf einem ähnlichen Niveau wie Mode 1, zeigt jedoch eine erhöhte Varianz. Die distanzbasierte Seed-Strategie erzeugt eine hohe Knotendichte im Bereich des Engpasses, um Konnektivität sicherzustellen. Je nach Seed-Platzierung kann dies entweder effizient verlaufen oder zu sehr vielen kollisionsanfälligen Kandidaten führen. Mode 3 (Max–Min) zeigt eine vergleichbare mittlere Anzahl an Kollisionschecks, weist jedoch die größte Varianz auf. Die globale Explorationsstrategie führt je nach Run entweder zu einer effizienten Umgehung des Engpasses oder zu einer ungünstigen Platzierung einzelner Knoten nahe an Hindernissen. Dadurch schwankt der Prüfaufwand stark zwischen sehr effizienten und sehr teuren Läufen. Mode 4 (start–goal corridor) liegt leicht über der Baseline und unterhalb der Seed-basierten Modi. Die gezielte Einschränkung des Samplings auf den Start–Ziel-Korridor fokussiert die Exploration auf den relevanten Durchgang, ohne eine starke lokale Überverdichtung zu erzeugen. Dadurch bleibt sowohl der Mittelwert als auch die Varianz der Kollisionschecks vergleichsweise gering.

**Pfadlänge**

Die Pfadlängen unterscheiden sich im WallTinyDoor-Benchmark moderat, zeigen jedoch eine klare Tendenz im Zusammenspiel zwischen Engpassstruktur und Samplingstrategie. Der schmale Durchgang erzwingt eine präzise Routenführung, wodurch ineffiziente Exploration direkt zu längeren Umwegen führt. Die Baseline weist eine vergleichsweise hohe mittlere Pfadlänge auf. Das uniforme Sampling erzeugt zwar valide Verbindungen durch den Engpass, jedoch ohne gezielte Verdichtung entlang der optimalen Passage. Dadurch entstehen häufiger indirekte Pfade mit zusätzlichen Richtungswechseln. Mode 1 (seed_gauss) reduziert die Pfadlänge gegenüber der Baseline leicht. Die lokale Verdichtung um Seeds verbessert die Konnektivität im Bereich des Engpasses und erlaubt etwas direktere Übergänge, ohne jedoch eine strikt optimale Linienführung zu erzwingen. Mode 2 (seed_dist) erzielt die kürzeste mittlere Pfadlänge. Die distanzbasierte Seed-Strategie führt zu einer kontrollierten, gleichmäßigen Nachverdichtung entlang des Engpasses und unterstützt eine sehr präzise Platzierung der Knoten. Dadurch entstehen besonders direkte und effiziente Pfade durch die schmale Öffnung. Mode 3 (Max–Min) weist die größte mittlere Pfadlänge auf und zeigt zudem eine erhöhte Streuung. Die globale Explorationsstrategie erzeugt eine geringe Knotendichte im Engpassbereich, wodurch die resultierenden Pfade gröber aufgelöst sind und häufiger Umwege enthalten. Einzelne Runs können dennoch gute Lösungen liefern, die Gesamttendenz bleibt jedoch ungünstiger. Mode 4 (start–goal corridor) erreicht eine Pfadlänge auf dem Niveau von Mode 1 und nahe bei Mode 2. Die Start–Ziel-Fokussierung begünstigt eine direkte Passage durch den Engpass, ohne jedoch die gleiche lokale Präzision wie Mode 2 zu erreichen. Dadurch entstehen kurze, aber nicht minimal optimale Pfade.

**Roadmapgröße**

Die Roadmapgröße zeigt im WallTinyDoor-Benchmark deutliche Unterschiede zwischen den Sampling-Modi. Der schmale Engpass erfordert eine präzise lokale Konnektivität, was sich direkt in der Anzahl der erzeugten Knoten widerspiegelt. Die Baseline erzeugt Roadmaps mittlerer Größe mit hoher Varianz. Das uniforme Sampling führt zu einer stark run-abhängigen Knotenzahl: In günstigen Fällen reichen wenige Knoten aus, um den Engpass zu durchdringen, in ungünstigen Läufen wächst die Roadmap deutlich an. Mode 1 (seed_gauss) erzeugt deutlich größere Roadmaps als die Baseline. Die lokale Verdichtung um Seeds führt zu einer hohen Knotendichte im Bereich des Engpasses, um eine zuverlässige Konnektivität sicherzustellen. Die Varianz bleibt hoch, da die Effektivität der Seeds stark vom jeweiligen Run abhängt. Mode 2 (seed_dist) weist die größte mittlere Roadmapgröße auf und zeigt zudem eine sehr große Streuung. Die distanzbasierte Seed-Strategie erzwingt eine intensive lokale Nachverdichtung entlang des Engpasses, was zu einer sehr hohen Anzahl an Knoten führt. Diese Strategie priorisiert Robustheit gegenüber struktureller Effizienz. Mode 3 (Max–Min) erzeugt die kleinsten Roadmaps unter allen Modi. Die globale Explorationsstrategie ergänzt nur wenige, strategisch platzierte Knoten und vermeidet eine starke lokale Überverdichtung im Engpass. Dadurch bleibt die Roadmap kompakt, allerdings auf Kosten einer geringeren lokalen Auflösung. Mode 4 (start–goal corridor) liegt hinsichtlich der Roadmapgröße zwischen Baseline und Mode 3. Die Einschränkung des Samplings auf den Start–Ziel-Korridor reduziert unnötige Exploration, erfordert jedoch zusätzliche Knoten zur präzisen Navigation durch den Engpass. Dadurch ergibt sich eine moderate Roadmapgröße mit vergleichsweise stabiler Varianz.

**Gesamtbewertung und Eignung für den Benchmark**

Der Benchmark WallTinyDoor ist durch einen sehr schmalen, lokalen Engpass geprägt, der eine hochpräzise und zuverlässige Knotensetzung erfordert. Reine Effizienz in Planungszeit oder Kollisionschecks ist hier kein ausreichendes Qualitätskriterium, da viele schnelle Lösungen den Engpass strukturell nicht robust erfassen, was sich in einer nur mittleren Erfolgsrate der Baseline widerspiegelt. Strategien mit kontrollierter Exploration oder gezielter Strukturierung erhöhen die Wahrscheinlichkeit, den Engpass korrekt zu erschließen, gehen jedoch häufig mit höherem Rechenaufwand, größerer Varianz oder stärkerem Overengineering einher. Insgesamt zeigt WallTinyDoor, dass bei extrem schmalen Engpässen Robustheit gegenüber Zufallseinflüssen wichtiger ist als reine Laufzeiteffizienz, was sich klar in den unterschiedlichen Erfolgsraten der Modi widerspiegelt.

## Evaluation mit Benchmark 3: WallTinyWideDoor

Der WallTinyWideDoor-Benchmark besitzt weiterhin eine Engpassstruktur, dieser ist jedoch deutlich länger als beim WallTinyDoor-Szenario. Dadurch steigt die Notwendigkeit hochpräziser lokaler Verdichtung. Alle fünf Sampling-Modi (Baseline sowie Mode 1–4) wurden auf diesem Benchmark jeweils mit 30 unabhängigen Planungsdurchläufen getestet und evaluiert. Die Balkendiagramme zeigen dabei Mittelwert und Varianz der betrachteten Metriken über alle erfolgreichen Läufe und erlauben eine quantitative Bewertung der Stabilität und Effizienz der einzelnen Modi. Die darunter dargestellten Roadmap- und Pfadvisualisierungen repräsentieren jeweils einen Best-Case pro Modus. Dieser Best-Case wird über eine gewichtete Mehrkriterienbewertung bestimmt, die Planungszeit, Pfadlänge und Roadmapgröße berücksichtigt, wobei extreme Ausreißer vorab über Quantilfilter ausgeschlossen werden. Die Visualisierungen dienen somit der qualitativen Einordnung typischer Lösungsstrukturen und sind nicht direkt mit den Mittelwerten der Balkendiagramme gleichzusetzen.

In [ ]:
benchmarks = [bench_m2]

df = run_suite(
    planner_class=EnhancedLazyPRM,
    benchmarks=benchmarks,
    configs=configs,
    runs=30,
    base_seed=2025,
    progress_every=10,
    bench_mode_map=None
)

summary = summarize(df)

for b in benchmarks:
    plot_benchmark_like_task1(summary, b.name, mode_order=MODE_ORDER)

SEED_MAP = extract_best_seed_map(df, prefer_seed_runs_for_mode12=True)

BENCH_ORDER = [b.name for b in benchmarks]
visualize_best_per_mode_and_benchmark(
    df=df,
    benchmarks=benchmarks,
    configs=configs,
    planner_class=EnhancedLazyPRM,
    lazyPRMVisualize=lazyPRMVisualize,
    nodeSize=80,
    mode_order=MODE_ORDER,
    bench_order=BENCH_ORDER,
    figsize=(7, 7),
    prefer_seed_runs_for_mode12=True,
)

### Diskussion der Ergebnisse

**Planungszeit**

Die Planungszeit zeigt moderate Unterschiede zwischen den Modi, wobei die Unterschiede nun konsistent mit einer langgezogenen Engpasspassage interpretiert werden müssen: Entscheidend ist weniger „Trefferwahrscheinlichkeit“ eines einzelnen Doors, sondern der Aufwand, über die gesamte Engpasslänge ausreichend Konnektivität zu erzeugen. Baseline liegt im unteren Bereich der Planungszeit. Das uniforme Sampling erzeugt ohne zusätzlichen Strategie-Overhead eine Roadmap, die den Engpass häufig mit überschaubarem Aufwand abdeckt. Gleichzeitig ist die Lösung stark zufallsgetrieben: je nachdem, wie gut die Uniform-Samples entlang der Passage landen, schwankt der notwendige Zusatzaufwand (moderate Varianz). Mode 1 (seed_gauss) zeigt eine leicht erhöhte Planungszeit gegenüber der Baseline. Lokale Verdichtung um Seeds kann zwar Konnektivität verbessern, ist bei einem langen Engpass aber häufig ineffizient: Verdichtung an wenigen Seed-Orten hilft nur punktuell und erzeugt zusätzlichen Prüfaufwand, ohne die gesamte Passage gleichmäßig zu schließen. Mode 2 (seed_dist) liegt nochmals etwas über Mode 1. Die kontrollierte distanzbasierte Seed-Erweiterung produziert strukturierte lokale Netze, führt im langen Engpass jedoch eher zu „Overbuilding“ in Teilbereichen, während andere Passagenabschnitte weiterhin zusätzliche Arbeit benötigen. Dadurch steigt der mittlere Planungsaufwand. Mode 3 (Max–Min) weist die höchste mittlere Planungszeit und die größte Streuung auf. Der zusätzliche Rechenaufwand durch Kandidatengenerierung/-bewertung fällt hier deutlich ins Gewicht. Gleichzeitig ist Max–Min nicht explizit entlang der Engpassrichtung geführt: Je nach Run investiert es Samples in global „leere“ Regionen statt in die kontinuierliche Passage, was Laufzeit und Varianz erhöht. Mode 4 (start–goal corridor) liegt in der Planungszeit wieder im Bereich der Baseline (tendenziell niedrig). Für einen langen Engpass ist das plausibel: die Korridorstrategie fokussiert die Sample- und Verbindungsarbeit entlang einer start–zielgerichteten Achse und unterstützt damit eine kontinuierliche Konnektivität über die Passage, ohne globalen Overhead.

**Kollisionschecks**

Die Anzahl der Kollisionsprüfungen unterscheidet sich zwischen den Sampling-Modi deutlich und reflektiert, wie effizient die jeweilige Strategie die langgezogene Engpasspassage strukturell abdeckt. Die Baseline zeigt eine moderate Anzahl an Kollisionschecks mit überschaubarer Varianz. Das uniforme Sampling verteilt Kandidaten über den gesamten Raum, wodurch zwar viele irrelevante Prüfungen entstehen, die längere Engpassstruktur jedoch nicht systematisch überverdichtet wird. Mode 1 (seed_gauss) verursacht mehr Kollisionschecks als die Baseline. Die lokale Verdichtung um Seeds führt bei einem langen Engpass dazu, dass einzelne Teilbereiche intensiv beprobt werden, während andere Abschnitte unstrukturiert bleiben. Die zusätzliche lokale Überprüfung erhöht den Prüfaufwand, ohne die gesamte Passage gleichmäßig abzudecken. Mode 2 (seed_dist) liegt auf einem ähnlichen Niveau wie Mode 1, leicht darüber. Die distanzbasierte Seed-Strategie erzeugt strukturiertere lokale Netze, führt im langgezogenen Engpass jedoch weiterhin zu einer segmentierten Nachverdichtung. Dadurch steigt die Anzahl kollisionsanfälliger Prüfungen entlang der Engpasswände. Mode 3 (Max–Min) weist die höchste mittlere Anzahl an Kollisionschecks und zugleich die größte Varianz auf. Die globale Explorationsstrategie generiert Kandidaten in schlecht abgedeckten Regionen, die bei einem langen Engpass häufig nahe an Hindernisrändern liegen. Je nach Run investiert Mode 3 entweder effizient entlang der Passage oder in ungünstige Randbereiche, was die starke Streuung erklärt. Mode 4 (start–goal corridor) erreicht die niedrigste Anzahl an Kollisionschecks und zeigt zugleich eine geringe Varianz. Die gezielte Einschränkung des Samplings auf den Start–Ziel-Korridor unterstützt eine kontinuierliche Abdeckung der Engpassrichtung und vermeidet unnötige Exploration seitlicher Hindernisbereiche.

**Pfadlänge**

Die Baseline erzeugt eine vergleichsweise lange mittlere Pfadlänge mit erhöhter Varianz. Das uniforme Sampling führt häufig zu ungleichmäßiger Abdeckung entlang der Engpassrichtung, sodass der Pfad aus mehreren indirekten Teilsegmenten zusammengesetzt wird. Mode 1 (seed_gauss) reduziert die Pfadlänge gegenüber der Baseline leicht. Die lokale Verdichtung verbessert die Konnektivität in Teilbereichen des Engpasses, führt jedoch nicht zu einer durchgehend gleichmäßigen Struktur über die gesamte Passage. Entsprechend bleibt der Pfad nur moderat kürzer. Mode 2 (seed_dist) liegt sehr nah bei Mode 1. Die kontrollierte distanzbasierte Erweiterung stabilisiert lokale Strukturen, erzielt im langen Engpass jedoch keinen zusätzlichen globalen Vorteil. Die Pfade bleiben ähnlich lang, da die Passage nicht durchgängig optimal abgedeckt wird. Mode 3 (Max–Min) weist die größte mittlere Pfadlänge und die größte Streuung auf. Die globale Explorationsstrategie erzeugt eine sehr grobe Abdeckung der Passage, wodurch Pfade häufig aus langen Kanten und Richtungswechseln bestehen. Einzelne Runs können gute Ergebnisse liefern, im Mittel bleibt die Pfadqualität jedoch deutlich schlechter. Mode 4 (start–goal corridor) erzielt die kürzesten und stabilsten Pfade. Die Start–Ziel-Fokussierung unterstützt eine kontinuierliche Knotenplatzierung entlang der Engpassrichtung und vermeidet unnötige seitliche Ausweichbewegungen. Dadurch entstehen nahezu geradlinige Pfade mit geringer Varianz.

**Roadmapgröße**

Die Baseline erzeugt Roadmaps mittlerer Größe mit hoher Varianz. Das uniforme Sampling verteilt Knoten über den gesamten Raum, sodass die Abdeckung des Engpasses stark run-abhängig ist. In einigen Läufen entstehen ausreichend viele Knoten entlang der Passage, in anderen bleibt die Struktur lückenhaft und erfordert zusätzliche Expansion. Mode 1 (seed_gauss) liegt größenmäßig nahe bei der Baseline, mit leicht erhöhter mittlerer Knotenzahl. Die lokale Verdichtung um Seeds führt zu zusätzlichen Knoten in Teilbereichen des Engpasses, ohne jedoch eine gleichmäßige Abdeckung über die gesamte Länge sicherzustellen. Die Varianz bleibt entsprechend hoch. Mode 2 (seed_dist) weist die größte mittlere Roadmapgröße und zugleich eine sehr große Streuung auf. Die distanzbasierte Seed-Strategie erzwingt eine intensive lokale Nachverdichtung, um Konnektivität entlang des Engpasses sicherzustellen. Bei einem langen Engpass führt dies zu starkem Overbuilding in einzelnen Segmenten, was die Knotenzahl deutlich erhöht. Mode 3 (Max–Min) erzeugt die kleinsten Roadmaps unter allen Modi. Durch die globale Explorationsstrategie werden nur wenige, strategisch platzierte Knoten ergänzt. Diese Abdeckung ist zwar strukturell effizient, jedoch nicht explizit entlang der Engpassrichtung optimiert, was sich in anderen Qualitätskriterien negativ auswirkt. Mode 4 (start–goal corridor) liegt hinsichtlich der Roadmapgröße zwischen Baseline und Mode 3. Die gezielte Einschränkung des Samplings auf den Start–Ziel-Korridor erzeugt eine kompakte, entlang der Passage ausgerichtete Struktur, ohne die starke Überverdichtung von Mode 2. Die Varianz bleibt moderat.

**Gesamtbewertung und Eignung für den Benchmark**

Der WallTinyWideDoor-Benchmark verdeutlicht, dass lange Engpässe weniger durch punktuelle Präzision als durch robuste, durchgängige Konnektivität entlang einer Richtung bestimmt sind. Strategien mit starker lokaler Verdichtung oder enger Start–Ziel-Fokussierung zeigen sich dabei wenig zuverlässig, da sie die Passage nur fragmentiert oder zu starr abdecken, was sich in sehr niedrigen Erfolgsraten äußert. Globale Explorationsansätze erhöhen die Wahrscheinlichkeit, den Engpass vollständig zu erfassen, gehen jedoch mit höherem Rechenaufwand, längeren Pfaden und größerer Varianz einher. Insgesamt offenbart der Benchmark einen klaren Zielkonflikt zwischen Robustheit gegenüber der Engpassgeometrie und Effizienz hinsichtlich Laufzeit, Pfadqualität und Roadmapgröße.

## Evaluation mit Benchmark 4: U-Shape

Der U-Shape-Benchmark stellt eine Umgebung mit einer ausgeprägten Sackgassen- bzw. Umgehungsstruktur dar. Start und Ziel sind räumlich relativ nahe beieinander, jedoch durch eine U-förmige Hindernisstruktur getrennt, sodass ein direkter Weg blockiert ist und eine gezielte Umfahrung erforderlich wird. Der Benchmark erfordert damit weniger globale Abdeckung, sondern vor allem die Fähigkeit, strukturbedingt irreführende direkte Verbindungen zu vermeiden und alternative, nicht-triviale Pfade zu finden. Alle fünf Sampling-Modi (Baseline sowie Mode 1–4) wurden auf diesem Benchmark jeweils mit 30 unabhängigen Planungsdurchläufen getestet und evaluiert. Die Balkendiagramme zeigen Mittelwert und Varianz der betrachteten Metriken über alle erfolgreichen Läufe. Die darunter dargestellten Roadmap- und Pfadvisualisierungen repräsentieren jeweils einen Best-Case, der über eine gewichtete Mehrkriterienbewertung aus Planungszeit, Pfadlänge und Roadmapgröße bestimmt wird, wobei extreme Ausreißer vorab über Quantilfilter ausgeschlossen werden. Die Visualisierungen dienen somit der qualitativen Einordnung typischer Lösungsstrukturen und sind nicht direkt mit den Mittelwerten der Balkendiagramme gleichzusetzen.

In [ ]:
benchmarks = [bench_m3]

df = run_suite(
    planner_class=EnhancedLazyPRM,
    benchmarks=benchmarks,
    configs=configs,
    runs=30,
    base_seed=2025,
    progress_every=10,
    bench_mode_map=None
)

summary = summarize(df)

for b in benchmarks:
    plot_benchmark_like_task1(summary, b.name, mode_order=MODE_ORDER)

SEED_MAP = extract_best_seed_map(df, prefer_seed_runs_for_mode12=True)

BENCH_ORDER = [b.name for b in benchmarks]
visualize_best_per_mode_and_benchmark(
    df=df,
    benchmarks=benchmarks,
    configs=configs,
    planner_class=EnhancedLazyPRM,
    lazyPRMVisualize=lazyPRMVisualize,
    nodeSize=80,
    mode_order=MODE_ORDER,
    bench_order=BENCH_ORDER,
    figsize=(7, 7),
    prefer_seed_runs_for_mode12=True,
)

### Diskussion der Ergebnisse

**Planungszeit**

Die Baseline weist eine geringe mittlere Planungszeit auf. Das uniforme Sampling verursacht keinen zusätzlichen Strategie-Overhead und erzeugt schnell eine Roadmap. Die geringe Laufzeit sagt jedoch nichts über die Qualität oder Zuverlässigkeit der Lösung aus, da viele Samples im topologisch falschen Bereich landen. Mode 1 (seed_gauss) zeigt die höchste mittlere Planungszeit sowie eine große Varianz. Die lokale Verdichtung um Seeds verstärkt im U-Shape-Szenario den falschen Fokus: Seeds entstehen häufig innerhalb oder nahe der U-Struktur, wodurch viele zusätzliche Kollisions- und Verbindungsprüfungen in einem nicht zielführenden Bereich durchgeführt werden. Mode 2 (seed_dist) liegt unterhalb von Mode 1, weist jedoch ebenfalls eine erhöhte Planungszeit und Varianz auf. Die distanzbasierte Seed-Strategie strukturiert lokale Bereiche zwar kontrollierter, bleibt jedoch weiterhin anfällig für die topologische Irreführung und erzeugt entsprechend zusätzlichen Aufwand. Mode 3 (Max–Min) erreicht eine sehr geringe mittlere Planungszeit, teilweise sogar unterhalb der Baseline. Die globale Explorationsstrategie identifiziert schnell schlecht abgedeckte Regionen außerhalb der U-Struktur und vermeidet eine übermäßige Verdichtung im topologisch falschen Raum. Dadurch reduziert sich der notwendige Planungsaufwand deutlich. Mode 4 weist zwar die niedrigste Planungszeit auf, findet jedoch keinen gültigen Pfad. Der Grund ist strukturell: Die U-Shape-Geometrie macht eine Umgehung erforderlich, während der Start–Ziel-Korridor gerade jene Regionen bevorzugt, die durch die U-Struktur blockiert bzw. topologisch irreführend sind.

**Kollisionschecks**

Die Anzahl der Kollisionsprüfungen spiegelt im U-Shape-Benchmark sehr deutlich wider, wie stark die jeweilige Samplingstrategie in topologisch falschen Bereichen exploriert. Da der direkte Weg zum Ziel blockiert ist, führen ungerichtete oder lokal fokussierte Strategien zu vielen ineffizienten Prüfungen. Die Baseline verursacht eine moderate Anzahl an Kollisionschecks mit relativ großer Varianz. Das uniforme Sampling verteilt Kandidaten über den gesamten Raum, wodurch sowohl zielführende als auch irrelevante Bereiche geprüft werden. Die Anzahl der Prüfungen bleibt insgesamt begrenzt, da keine gezielte Verdichtung stattfindet. Mode 1 (seed_gauss) weist die höchste mittlere Anzahl an Kollisionschecks sowie eine große Streuung auf. Die lokale Verdichtung um Seeds verstärkt im U-Shape-Szenario den falschen Fokus: Seeds entstehen häufig innerhalb oder nahe der U-Struktur, wodurch viele Knoten- und Kantenkandidaten in kollisionsanfälligen Bereichen erzeugt werden. Mode 2 (seed_dist) liegt unterhalb von Mode 1, zeigt jedoch ebenfalls eine hohe Varianz. Die distanzbasierte Seed-Strategie strukturiert lokale Regionen kontrollierter, bleibt aber weiterhin anfällig für die topologische Irreführung und erzeugt entsprechend viele Prüfungen in nicht zielführenden Bereichen. Mode 3 (Max–Min) verursacht die geringste Anzahl an Kollisionschecks unter den erfolgreichen Modi. Die globale Explorationsstrategie identifiziert früh schlecht abgedeckte Regionen außerhalb der U-Struktur und vermeidet eine starke Verdichtung im blockierten Innenbereich. Dadurch sinkt sowohl der Mittelwert als auch die Varianz der Kollisionschecks deutlich.

**Pfadlänge**

Die Baseline erzeugt eine vergleichsweise große mittlere Pfadlänge mit hoher Varianz. Das uniforme Sampling findet zwar Umgehungen, diese sind jedoch häufig indirekt und bestehen aus mehreren grob verbundenen Teilsegmenten. Mode 1 (seed_gauss) reduziert die Pfadlänge gegenüber der Baseline leicht. Die lokale Verdichtung um Seeds verbessert die Konnektivität in Teilbereichen, bleibt jedoch häufig im topologisch falschen Innenbereich der U-Struktur verankert. Dadurch bleiben die Pfade länger und variabel. Mode 2 (seed_dist) erzielt die kürzeste mittlere Pfadlänge unter den erfolgreichen Modi. Die kontrollierte distanzbasierte Erweiterung führt zu einer stabileren, gleichmäßigeren Abdeckung der relevanten Umgehungsregionen, was direktere und effizientere Pfade ermöglicht. Mode 3 (Max–Min) weist wieder eine größere mittlere Pfadlänge und hohe Varianz auf. Zwar gelingt es der globalen Exploration, den korrekten Umweg zuverlässig zu finden, die geringe Knotendichte führt jedoch zu gröberen Pfadverläufen mit längeren Kanten.

**Roadmapgröße**

Da der direkte Weg blockiert ist, führt ineffiziente Exploration schnell zu unnötigem Wachstum der Roadmap. Die Baseline erzeugt kleine bis mittlere Roadmaps mit moderater Varianz. Das uniforme Sampling ergänzt nur wenige Knoten, wodurch die Struktur kompakt bleibt. Diese Effizienz geht jedoch zulasten einer gezielten Abdeckung des notwendigen Umwegs. Mode 1 (seed_gauss) erzeugt deutlich größere Roadmaps und weist eine hohe Varianz auf. Die lokale Verdichtung um Seeds verstärkt die Exploration im topologisch falschen Bereich der U-Struktur, was zu vielen zusätzlichen, strukturell wenig hilfreichen Knoten führt. Mode 2 (seed_dist) liegt etwas unterhalb von Mode 1, zeigt jedoch ebenfalls große Roadmaps mit sehr hoher Streuung. Die distanzbasierte Erweiterung strukturiert lokale Bereiche kontrollierter, bleibt jedoch anfällig für falsche Schwerpunktsetzung innerhalb der U-Form und erzeugt entsprechend viele Knoten. Mode 3 (Max–Min) erzeugt mit Abstand die kleinsten und stabilsten Roadmaps unter den erfolgreichen Modi. Die globale Explorationsstrategie ergänzt nur wenige, gezielt platzierte Knoten außerhalb der blockierten Region und vermeidet lokales Overbuilding. Dadurch bleibt die Roadmap sehr kompakt.

**Gesamtbewertung und Eignung für den Benchmark**

Der U-Shape-Benchmark ist topologisch irreführend, da der direkte Start–Ziel-Weg blockiert ist und eine explizite globale Umgehung erforderlich wird. Dies spiegelt sich klar in den Erfolgsraten wider: Während Baseline, Mode 1 und Mode 3 alle 30/30 erfolgreiche Runs erreichen, scheitert Mode 4 vollständig (0/30), da die Start–Ziel-Fokussierung den notwendigen Umweg systematisch verfehlt; Mode 2 ist mit 9/30 nur eingeschränkt robust. Lokale Seed-basierte Strategien erzeugen zwar oft kurze Pfade, investieren jedoch viel Aufwand im topologisch falschen Bereich, was sich in hoher Planungszeit, vielen Kollisionschecks und großen Roadmaps äußert. Insgesamt zeigt der Benchmark deutlich, dass bei irreführender Geometrie globale Exploration wichtiger ist als lokale Verdichtung oder reine Richtungsführung, und dass Erfolgsrate hier das entscheidende Qualitätskriterium darstellt.

## Evaluation mit Benchmark 5: Snail

Der Snail-Benchmark stellt eine besonders anspruchsvolle Umgebung dar, die durch eine lange, schmale, spiralförmige Passage geprägt ist. Start und Ziel liegen zwar räumlich nicht extrem weit auseinander, sind jedoch durch die verschachtelte Hindernisstruktur nur über einen sehr spezifischen, winding Pfad miteinander verbunden. Der Benchmark erfordert daher weniger lokale Engstellenauflösung im klassischen Sinn, sondern vielmehr eine globale, strukturtreue Exploration entlang einer stark eingeschränkten Geometrie. Direkte oder lokal verdichtende Strategien laufen hier Gefahr, sich frühzeitig in Sackgassen oder irrelevanten Bereichen festzufahren. Alle fünf Sampling-Modi (Baseline sowie Mode 1–4) wurden auf diesem Benchmark jeweils mit 30 unabhängigen Planungsdurchläufen getestet und evaluiert. Die Balkendiagramme zeigen Mittelwert und Varianz der betrachteten Metriken über erfolgreiche Läufe. Da mehrere Modi in diesem Benchmark keine oder nur extrem wenige gültige Lösungen finden, sind einige Balken leer bzw. nicht aussagekräftig. Die darunter dargestellten Roadmap- und Pfadvisualisierungen zeigen jeweils entweder einen Best-Case (falls vorhanden) oder einen Fallback-Zustand, wenn kein gültiger Pfad gefunden wurde. Der Best-Case wird – analog zu den vorherigen Benchmarks – über eine gewichtete Mehrkriterienbewertung aus Planungszeit, Pfadlänge und Roadmapgröße bestimmt, wobei extreme Ausreißer über Quantilfilter ausgeschlossen werden.

In [ ]:
benchmarks = [bench_m4]

df = run_suite(
    planner_class=EnhancedLazyPRM,
    benchmarks=benchmarks,
    configs=configs,
    runs=30,
    base_seed=2025,
    progress_every=10,
    bench_mode_map=None
)

summary = summarize(df)

for b in benchmarks:
    plot_benchmark_like_task1(summary, b.name, mode_order=MODE_ORDER)

SEED_MAP = extract_best_seed_map(df, prefer_seed_runs_for_mode12=True)

BENCH_ORDER = [b.name for b in benchmarks]
visualize_best_per_mode_and_benchmark(
    df=df,
    benchmarks=benchmarks,
    configs=configs,
    planner_class=EnhancedLazyPRM,
    lazyPRMVisualize=lazyPRMVisualize,
    nodeSize=80,
    mode_order=MODE_ORDER,
    bench_order=BENCH_ORDER,
    figsize=(7, 7),
    prefer_seed_runs_for_mode12=True,
)

### Diskussion der Ergebnisse

Der Snail-Benchmark stellt eine extrem strukturierte, verschachtelte Engpassgeometrie dar, die eine konsequente globale Exploration entlang einer winding corridor-Struktur erfordert. Lokale oder stark gerichtete Strategien scheitern hier strukturell. Das uniforme Sampling exploriert den Raum global und ist dadurch grundsätzlich in der Lage, den spiralförmigen Korridor schrittweise zu erschließen. Der Aufwand bleibt stabil, da keine zusätzliche Kandidatenbewertung oder lokale Überverdichtung erfolgt. Mode 3 ist der einzige Enhancement-Modus, der den Snail-Benchmark zuverlässig lösen kann, da die Max–Min-Strategie gezielt schlecht abgedeckte Regionen entlang der verschachtelten Struktur identifiziert und fortschreitend erschließt.

Mode 1 (seed_gauss), Mode 2 (seed_dist) und Mode 4 (start–goal corridor) finden keinen gültigen Pfad. Die seed-basierten Modi fokussieren sich lokal auf bereits bekannte Regionen und scheitern daran, die globale Spiralstruktur schrittweise zu durchdringen. Mode 4 ist durch seine Start–Ziel-Fokussierung fundamental fehlgeleitet, da der direkte Korridor vollständig blockiert ist und der notwendige Umweg mehrfach die Richtung wechselt.

**Gesamtbewertung und Eignung für den Benchmark**

Der Snail-Benchmark stellt eine extrem anspruchsvolle, verschachtelte Engpassgeometrie dar, bei der der Zielpunkt nur über einen langen, mehrfach gewundenen Korridor erreichbar ist. Dies spiegelt sich unmittelbar in den Erfolgsraten wider: Während die Baseline mit uniformem Sampling nur 1 von 30 Runs erfolgreich abschließen kann, ist Mode 3 (Max–Min) der einzige Modus mit substantieller Erfolgsrate (23/30), da er gezielt schlecht abgedeckte Regionen entlang der Spiralstruktur exploriert. Lokale Seed-basierte Strategien sowie die Start–Ziel-Korridorstrategie scheitern vollständig (0/30), da sie entweder im bereits bekannten Raum verharren oder durch die topologisch irreführende Geometrie fehlgeleitet werden. Insgesamt zeigt der Snail-Benchmark sehr deutlich, dass bei stark verschachtelten, nicht-monotonen Engpassstrukturen robuste globale Exploration entscheidend ist, selbst wenn sie mit höherem Rechenaufwand einhergeht.

# 2DoF Punktroboter

Zur ergänzenden qualitativen Analyse der Node-Enhancementstrategien wird der 2-DOF-Punktroboter betrachtet. Dieses Robotermodell stellt die einfachste untersuchte Systemklasse dar und dient als Referenz, um die Wirkungsweise der unterschiedlichen Sampling-Modi anschaulich zu untersuchen. Der Punktroboter bewegt sich rein translational im zweidimensionalen Arbeitsraum, sodass Konfigurationsraum und Arbeitsraum identisch sind. Obwohl der Roboter kinematisch als Punkt modelliert ist, wird ein endlicher Roboter­radius von 0,25 Längeneinheiten + einem Sicherheitsabstand von 0,05 Längeneinheiten in der Kollisionsprüfung berücksichtigt. Die Hindernisse werden somit effektiv um diesen Radius aufgeblasen, sodass auch für den Punktroboter realistische Abstands- und Kollisionsbedingungen gelten. Im Vergleich zum 2-DOF-Planarroboter entfallen jedoch rotationsabhängige Freiheitsgrade und kinematische Kopplungen, wodurch die geometrischen Effekte der Sampling-Strategien besonders klar sichtbar werden. Der Punktroboter eignet sich daher hervorragend als Analyse-Baseline, um Unterschiede in Samplingdichte, Roadmapstruktur und Pfadführung isoliert zu untersuchen, ohne dass diese durch komplexe Gelenkkinematik oder aufwendige Kollisionsmodelle überlagert werden.

## Konfiguration

In diesem Abschnitt wird die Notebook-Umgebung so konfiguriert, dass Codeänderungen automatisch übernommen werden und Animationen direkt im Notebook dargestellt werden können. Anschließend werden die Konfigurationen des Lazy-PRM für alle Sampling-Modi neu erzeugt, wobei erstmals ein expliziter Roboterradius sowie eine Sicherheitsmarge berücksichtigt werden. Auf dieser Basis werden alle Benchmarks mit entsprechend aufgeblasenen Hindernissen neu instanziiert, sodass die nachfolgenden Evaluationen und Animationen konsistent die Robotershape in der Kollisionsprüfung abbilden.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib as mpl
mpl.rcParams["animation.html"] = "jshtml"
mpl.rcParams["animation.embed_limit"] = 256

from point_robot_anim import animate_benchmarks_from_seedmap

# Aus deiner Evaluation-Auslagerung:
from eval_task2 import run_suite, extract_best_seed_map

# Dein Planner:
from IPLazyPRM_Task2 import EnhancedLazyPRM

In [ ]:
# Config für 2DoF Punktroboter
# Abschnitt 2: Roboter mit Radius + Safety
base_config["robotRadius"]  = 0.25
base_config["safetyMargin"] = 0.05

# configs neu snapshotten (dict(base_config, ...) ist sonst noch der alte Stand!)
cfg_base = dict(base_config, enhanceMode="baseline_uniform")
cfg_m1   = dict(base_config, enhanceMode="mode1_seed_gauss", seedSigma=4.0, seedTries=5)
cfg_m2   = dict(base_config, enhanceMode="mode2_seed_dist", seedMaxStep=4.0, seedBeta=0.9, seedTries=5)
cfg_m3   = dict(base_config, enhanceMode="mode3_max_min", dispersionCandidates=5)
cfg_m4   = dict(base_config, enhanceMode="mode4_start_goal_corr",
                corridorSigma=3.0, corridorAlongSigma=0.6, corridorTries=6)

configs = {
    "baseline_uniform":       cfg_base,
    "mode1_seed_gauss":       cfg_m1,
    "mode2_seed_dist":        cfg_m2,
    "mode3_max_min":          cfg_m3,
    "mode4_start_goal_corr":  cfg_m4,
}

ROBOT_RADIUS = base_config["robotRadius"]
SAFETY       = base_config["safetyMargin"]
CLEARANCE    = ROBOT_RADIUS + SAFETY  # 0.30

# Benchmarks neu erstellen
# CircleField
bench_baseline = make_circle_field_benchmark(
    clearance=CLEARANCE,
    name="CircleField",
    bounds=(0.0, 22.0, 0.0, 22.0),
    start=(2, 20),
    goal=(20, 2),
    n_circles=40,
    r_min=1.3,
    r_max=2.2,
    min_center_dist=1.0,
    inner_keepout=2.2,
    edge_overshoot=2.8,   # mehr Randüberhang
    seed=7,
)

# WallTinyDoor
bench_m1 = make_mode1_wall_tiny_door_closed22(
    clearance=CLEARANCE,
    extra_block=True
)

# WallTinyWideDoor
bench_m2 = make_mode2_wall_tiny_wide_door(
    clearance=CLEARANCE,
    extra_block=True
)

# U-Shape
bench_m3 = make_mode3_u_shape_benchmark(
    clearance=CLEARANCE,
    name="U-Shape",
    bounds=(0.0, 22.0, 0.0, 22.0),
)

# Snail
bench_m4 = make_mode4_snail(
    clearance=CLEARANCE,
    name="Snail",
    bounds=(0.0, 22.0, 0.0, 22.0),
)

## Benchmark 1: Baseline_CircleField

Die dargestellten Animationen zeigen für jeden Sampling-Modus einen exemplarischen, zuvor berechneten Pfad innerhalb der aufgebauten Roadmap im Benchmark CircleField. Mit der Berücksichtigung der Robotershape sinkt zunächst die Erfolgsrate bei allen Modi deutlich. Während im idealisierten Punktmodell nahezu alle Strategien zuverlässig Pfade finden, führt der reduzierte freie Raum durch die aufgeblasenen Hindernisse dazu, dass unzureichend strukturierte oder ungünstig verdichtete Roadmaps häufiger keine gültige Verbindung mehr erzeugen. Besonders auffällig ist dies bei lokal stark verdichtenden Strategien, deren Erfolgsraten nun deutlich stärker auseinanderfallen. Die Planungszeit steigt insgesamt an und zeigt größere Varianz. Dies ist eine direkte Folge der erhöhten Anzahl an Kollisionschecks, da Knoten und Kanten, die im radiuslosen Modell noch valide waren, nun häufiger mit aufgeblasenen Hindernissen kollidieren. Unterschiede zwischen den Modi werden dadurch klarer sichtbar: Strategien mit kontrollierter Strukturierung oder sparsamer Knotenerzeugung reagieren robuster, während ungerichtetes oder stark lokales Sampling mehr Prüfaufwand erzeugt. Bei der Pfadlänge zeigt sich ein qualitativer Effekt: Mit Robotershape werden Pfade im Mittel länger und variabler. Größere Sicherheitsabstände erzwingen zusätzliche Ausweichbewegungen, sodass grob aufgelöste Roadmaps (z. B. mit wenigen, weit auseinanderliegenden Knoten) ihre Nachteile stärker offenbaren. Pfadqualität wird damit sensibler gegenüber der tatsächlichen Roadmapstruktur und nicht mehr allein durch geometrische Durchgänge bestimmt. Auch die Roadmapgröße verändert ihre Aussagekraft. Während im Punktmodell kleine Roadmaps oft ausreichend sind, benötigen Strategien mit Robotershape eine feinere lokale Auflösung, um sichere Verbindungen mit ausreichendem Abstand zu Hindernissen zu gewährleisten. Sehr kompakte Roadmaps verlieren dadurch an Robustheit, während moderat verdichtete Strukturen stabilere Lösungen liefern.

In [ ]:
BENCH_LIST = [
    ("bench_baseline", bench_baseline),
]

benchmarks = [b for _, b in BENCH_LIST]

df = run_suite(
    planner_class=EnhancedLazyPRM,
    benchmarks=benchmarks,
    configs=configs,
    runs=30,
    base_seed=2025,
    progress_every=10,
    bench_mode_map=None
)

# --- NEU: Summary + Balkendiagramme wie in der Evaluation ---
summary = summarize(df)

for b in benchmarks:
    plot_benchmark_like_task1(summary, b.name, mode_order=MODE_ORDER)

# --- SeedMap aus denselben Runs ziehen ---
SEED_MAP = extract_best_seed_map(df, prefer_seed_runs_for_mode12=True)
print("SEED_MAP entries:", len(SEED_MAP))

# --- Animation ---
animate_benchmarks_from_seedmap(
    bench_list=BENCH_LIST,
    mode_order=MODE_ORDER,
    seed_map=SEED_MAP,
    configs=configs,
    planner_class=EnhancedLazyPRM,
    interp_step=0.5,
    fig_size=(7, 7),
    obstacle_alpha=0.25,
    robot_visual="robot",      # oder "clearance"
    draw_robot_fill=False,
    robot_alpha=0.25,
    show_debug=True,
)


## Benchmark 2: WallTinyDoor

Die dargestellten Animationen zeigen für jeden Sampling-Modus einen exemplarischen, zuvor berechneten Pfad innerhalb der aufgebauten Roadmap im Benchmark WallTinyDoor. Mit der Berücksichtigung der Robotershape verändert sich das Verhalten der Sampling-Strategien deutlich stärker als im radiuslosen Punktmodell. Durch die effektive Verengung des ohnehin sehr schmalen Engpasses sinkt die Erfolgsrate bei allen Modi, da zufällige oder grob strukturierte Knotensetzungen nun wesentlich seltener ausreichend Abstandsreserven besitzen, um eine gültige Verbindung durch den Durchgang zu ermöglichen. Besonders lokal strukturierende Verfahren reagieren empfindlich auf diese Verschärfung, wodurch sich die Erfolgsraten der Modi stärker auseinanderziehen. Auffällig ist, dass die Planungszeiten insgesamt sinken, was jedoch keinen Effizienzgewinn darstellt. Vielmehr ist dieser Effekt auf ein häufigeres frühes Scheitern der Planung zurückzuführen: Viele Runs brechen ab, bevor eine große Roadmap aufgebaut wird, wodurch weniger Erweiterungs-, Verbindungs- und Kollisionsprüfungen durchgeführt werden. Dieser Effekt zeigt sich besonders deutlich bei Mode 1, dessen Profil sich gegenüber dem radiuslosen Fall stark verändert. Während Mode 1 zuvor durch hohe Laufzeiten infolge intensiver lokaler Verdichtung auffiel, entstehen mit Robotershape deutlich kleinere Roadmaps, geringere Kollisionschecks und kurze Laufzeiten – allerdings auf Kosten einer stark reduzierten Erfolgsrate. Auch die Pfadlängen verändern ihre Aussagekraft. Mit Robotershape erzwingen die vergrößerten Sicherheitsabstände eine präzisere Führung durch den Engpass, sodass Pfade bei erfolgreicher Planung weniger durch lokale Nachbesserung, sondern stärker durch die grundsätzliche Struktur der Roadmap bestimmt werden. Strategien mit unzureichender lokaler Auflösung verlieren hier an Qualität, selbst wenn sie zuvor akzeptable Lösungen liefern konnten. Schließlich zeigt sich bei der Roadmapgröße, dass kompakte Strukturen im WallTinyDoor mit Robotershape nicht mehr automatisch vorteilhaft sind. Während im punktförmigen Modell kleine Roadmaps häufig ausreichten, benötigen erfolgreiche Strategien nun eine gezielte, aber kontrollierte lokale Auflösung im Engpassbereich. Zu starke lokale Verdichtung führt hingegen schnell zu Overengineering und erhöhtem Prüfaufwand, ohne die Robustheit proportional zu steigern.

In [ ]:
BENCH_LIST = [
    ("bench_m1", bench_m1),
]

benchmarks = [b for _, b in BENCH_LIST]

df = run_suite(
    planner_class=EnhancedLazyPRM,
    benchmarks=benchmarks,
    configs=configs,
    runs=30,
    base_seed=2025,
    progress_every=10,
    bench_mode_map=None
)

# --- NEU: Summary + Balkendiagramme wie in der Evaluation ---
summary = summarize(df)

for b in benchmarks:
    plot_benchmark_like_task1(summary, b.name, mode_order=MODE_ORDER)

# --- SeedMap aus denselben Runs ziehen ---
SEED_MAP = extract_best_seed_map(df, prefer_seed_runs_for_mode12=True)
print("SEED_MAP entries:", len(SEED_MAP))

# --- Animation ---
animate_benchmarks_from_seedmap(
    bench_list=BENCH_LIST,
    mode_order=MODE_ORDER,
    seed_map=SEED_MAP,
    configs=configs,
    planner_class=EnhancedLazyPRM,
    interp_step=0.5,
    fig_size=(7, 7),
    obstacle_alpha=0.25,
    robot_visual="robot",      # oder "clearance"
    draw_robot_fill=False,
    robot_alpha=0.25,
    show_debug=True,
)


## Benchmark 3: WallTinyWideDoor

Die dargestellten Ergebnisse vergleichen die Sampling-Modi im Benchmark WallTinyWideDoor für das idealisierte Punktmodell und für die Variante mit berücksichtigter Robotershape. Mit Einführung eines Roboterradius sinkt die Erfolgsrate bei allen Strategien deutlich, da der zuvor vergleichsweise breite Durchgang durch die aufgeblasenen Hindernisse zu einem stark eingeschränkten zulässigen Konfigurationsraum wird. Während im radiuslosen Modell mehrere Modi zuverlässig gültige Verbindungen finden, scheitern mit Robotershape nahezu alle lokal fokussierten oder uniformen Strategien; lediglich der global explorierende Max–Min-Ansatz erzielt noch vereinzelt erfolgreiche Runs. Die gemessenen Planungszeiten sinken insgesamt, was primär auf frühzeitiges Abbrechen bei ausbleibender Lösungsfindung zurückzuführen ist und nicht auf effizientere Planung. Die Pfadlänge verändert sich hingegen nur moderat, da der geometrische Durchgang weiterhin ähnlich verläuft und sich die Haupttrajektorie trotz reduzierten freien Raums kaum verschiebt. Deutlich ausgeprägter sind die Unterschiede in der Roadmapgröße: Insbesondere Mode 1 und Mode 2 erzeugen mit Robotershape sehr große Roadmaps, ohne daraus einen entsprechenden Nutzen zu ziehen. Die starke lokale Verdichtung in der Nähe ungünstiger oder kollisionsnaher Bereiche führt zu erhöhtem Knotenzuwachs, während valide Verbindungen durch den Engpass dennoch ausbleiben. Insgesamt verdeutlicht der Benchmark, dass mit Robotershape nicht die Anzahl der Knoten, sondern deren globale Platzierung entscheidend wird und lokal verdichtende Strategien in stark eingeschränkten Durchgangsszenarien strukturell benachteiligt sind.

In [ ]:
BENCH_LIST = [
    ("bench_m2", bench_m2),
]

benchmarks = [b for _, b in BENCH_LIST]

df = run_suite(
    planner_class=EnhancedLazyPRM,
    benchmarks=benchmarks,
    configs=configs,
    runs=30,
    base_seed=2025,
    progress_every=10,
    bench_mode_map=None
)

# --- NEU: Summary + Balkendiagramme wie in der Evaluation ---
summary = summarize(df)

for b in benchmarks:
    plot_benchmark_like_task1(summary, b.name, mode_order=MODE_ORDER)

# --- SeedMap aus denselben Runs ziehen ---
SEED_MAP = extract_best_seed_map(df, prefer_seed_runs_for_mode12=True)
print("SEED_MAP entries:", len(SEED_MAP))

# --- Animation ---
animate_benchmarks_from_seedmap(
    bench_list=BENCH_LIST,
    mode_order=MODE_ORDER,
    seed_map=SEED_MAP,
    configs=configs,
    planner_class=EnhancedLazyPRM,
    interp_step=0.5,
    fig_size=(7, 7),
    obstacle_alpha=0.25,
    robot_visual="robot",      # oder "clearance"
    draw_robot_fill=False,
    robot_alpha=0.25,
    show_debug=True,
)

## Benchmark 4: U-Shape

Die Ergebnisse für den Benchmark U-Shape zeigen im Vergleich zwischen Punktmodell und Berücksichtigung der Robotershape insgesamt nur geringe Veränderungen. Sowohl die Anzahl der erfolgreichen Runs als auch die grundlegende Leistungscharakteristik der einzelnen Sampling-Modi bleiben weitgehend stabil. Der reduzierte freie Raum durch die aufgeblasenen Hindernisse beeinflusst die Lösbarkeit des Problems nur marginal, da die zugrunde liegende U-förmige Struktur weiterhin einen klar definierten und ausreichend breiten Durchgang bietet. Entsprechend ändern sich auch Kollisionschecks, Pfadlängen und Roadmapgrößen nur geringfügig. Die beobachtete Abnahme der Planungszeiten ist primär auf frühzeitige Abbrüche einzelner nicht erfolgreicher Läufe zurückzuführen und stellt keinen Effizienzgewinn der Verfahren dar. Insgesamt verdeutlicht der U-Shape-Benchmark, dass Szenarien mit eindeutiger geometrischer Führung und ohne enge Engpässe vergleichsweise robust gegenüber der Berücksichtigung einer Robotershape sind und sich Unterschiede zwischen den Sampling-Strategien hier nur abgeschwächt manifestieren.

In [ ]:
BENCH_LIST = [
    ("bench_m3", bench_m3),
]

benchmarks = [b for _, b in BENCH_LIST]

df = run_suite(
    planner_class=EnhancedLazyPRM,
    benchmarks=benchmarks,
    configs=configs,
    runs=30,
    base_seed=2025,
    progress_every=10,
    bench_mode_map=None
)

# --- NEU: Summary + Balkendiagramme wie in der Evaluation ---
summary = summarize(df)

for b in benchmarks:
    plot_benchmark_like_task1(summary, b.name, mode_order=MODE_ORDER)

# --- SeedMap aus denselben Runs ziehen ---
SEED_MAP = extract_best_seed_map(df, prefer_seed_runs_for_mode12=True)
print("SEED_MAP entries:", len(SEED_MAP))

# --- Animation ---
animate_benchmarks_from_seedmap(
    bench_list=BENCH_LIST,
    mode_order=MODE_ORDER,
    seed_map=SEED_MAP,
    configs=configs,
    planner_class=EnhancedLazyPRM,
    interp_step=0.5,
    fig_size=(7, 7),
    obstacle_alpha=0.25,
    robot_visual="robot",      # oder "clearance"
    draw_robot_fill=False,
    robot_alpha=0.25,
    show_debug=True,
)

## Benchmark 5: Snail

Die Ergebnisse für den Benchmark Snail verdeutlichen eine ausgeprägte Sensitivität gegenüber der Berücksichtigung der Robotershape. Bereits im Punktmodell zeigt sich, dass nahezu alle Sampling-Modi am stark gewundenen, schmalen Korridor scheitern und lediglich der global explorierende Max–Min-Ansatz in der Lage ist, mit relevanter Erfolgsrate gültige Pfade zu finden. Mit Einführung eines Roboterradius verschärft sich diese Situation weiter: Die Anzahl erfolgreicher Runs sinkt deutlich, während alle übrigen Strategien vollständig versagen. Ursache hierfür ist die Kombination aus langer, stark eingeschränkter Passage und mehrfachen Richtungswechseln, die eine kontinuierliche, präzise Platzierung von Knoten entlang des freien Raums erfordert. Die Planungszeiten nehmen mit Robotershape ab, was primär auf frühzeitige Abbrüche bei ausbleibender Lösungsfindung zurückzuführen ist. Kollisionschecks, Pfadlängen und Roadmapgrößen bleiben dabei im Wesentlichen auf dem Niveau des Punktmodells, da erfolgreiche Lösungen weiterhin ausschließlich durch Mode 3 bestimmt werden. Insgesamt bestätigt der Snail-Benchmark, dass stark verschachtelte und topologisch anspruchsvolle Umgebungen mit Robotershape ein reines Explorationsproblem darstellen, bei dem globale Abdeckung entscheidend ist und lokal fokussierte oder strukturarme Sampling-Strategien systematisch an ihre Grenzen stoßen.

In [ ]:
BENCH_LIST = [
    ("bench_m4", bench_m4),
]

benchmarks = [b for _, b in BENCH_LIST]

df = run_suite(
    planner_class=EnhancedLazyPRM,
    benchmarks=benchmarks,
    configs=configs,
    runs=30,
    base_seed=2025,
    progress_every=10,
    bench_mode_map=None
)

# --- NEU: Summary + Balkendiagramme wie in der Evaluation ---
summary = summarize(df)

for b in benchmarks:
    plot_benchmark_like_task1(summary, b.name, mode_order=MODE_ORDER)

# --- SeedMap aus denselben Runs ziehen ---
SEED_MAP = extract_best_seed_map(df, prefer_seed_runs_for_mode12=True)
print("SEED_MAP entries:", len(SEED_MAP))

# --- Animation ---
animate_benchmarks_from_seedmap(
    bench_list=BENCH_LIST,
    mode_order=MODE_ORDER,
    seed_map=SEED_MAP,
    configs=configs,
    planner_class=EnhancedLazyPRM,
    interp_step=0.5,
    fig_size=(7, 7),
    obstacle_alpha=0.25,
    robot_visual="robot",      # oder "clearance"
    draw_robot_fill=False,
    robot_alpha=0.25,
    show_debug=True,
)

# 2Dof Planarroboter

Im Gegensatz zum zuvor betrachteten 2-DoF-Punktroboter bewegt sich der 2-DoF-Planarroboter nicht in einem kartesischen Arbeitsraum, sondern in einem zweidimensionalen Konfigurationsraum aus Gelenkwinkeln. Jeder Konfigurationspunkt beschreibt dabei eine vollständige Roboterpose, während geometrische Eigenschaften wie Nähe zu Hindernissen erst indirekt über Vorwärtskinematik und Kollisionsprüfung bewertet werden können. Die Struktur des Planungsproblems unterscheidet sich damit grundlegend von der Punktrobotervariante. Durch die explizite Modellierung der Robotergeometrie werden Kollisionstests deutlich aufwendiger. Während beim Punktroboter einfache Punkt-Hindernis-Tests ausreichen, müssen beim Planarroboter mehrere Robotersegmente gegen alle Hindernisse geprüft werden.  Durch die explizite Modellierung der Robotergeometrie verändert sich die Struktur des Planungsproblems grundlegend. Kollisionstests sind deutlich aufwendiger als beim Punktroboter, da mehrere starre Segmente des Manipulators gegen sämtliche Hindernisse geprüft werden müssen. Gleichzeitig entstehen im Konfigurationsraum komplexe, nichtlinear geformte Hindernisregionen (gelb gekennzeichnet), die selbst bei einfachen Arbeitsraumgeometrien eine stark strukturierte Topologie aufweisen. Diese wird in den Animationen durch die explizite Darstellung des Konfigurationsraums visualisiert, wobei die in den Arbeitsraum projizierten Hindernisse als verbotene Regionen im C-Space markiert sind. Für die Bewertung der Sampling-Modi werden – analog zu den Punktroboterszenarien – erneut identische Metriken herangezogen, darunter die Anzahl erfolgreicher Runs, Planungszeit, Kollisionschecks, Pfadlänge und Roadmapgröße. Die Analyse beschränkt sich dabei auf zwei repräsentative Benchmarks: Planar_Easy, das eine vergleichsweise gutartige Konfigurationsraumstruktur besitzt, sowie Planar_Passage, das durch schmale Durchgänge und eine stark eingeschchränkte Konnektivität im C-Space gekennzeichnet ist. Durch diese Gegenüberstellung lässt sich untersuchen, inwieweit sich die beobachteten Eigenschaften der Sampling-Strategien aus dem Punktrobotermodell auf den deutlich komplexeren Fall eines planar artikulierten Roboters übertragen lassen.

## Konfiguration

Im folgenden Abschnitt werden gemeinsame Basis-Konfiguration für den Planar-Lazy-PRM sowie die mode-spezifischen Parameter definiert und daraus für alle Sampling-Modi konsistente Konfigurationsobjekte erzeugt. Anschließend werden pro Benchmark kollisionsfreie Start-/Zielkonfigurationen validiert und als Overrides in jede Mode-Config injiziert, sodass alle Modi unter identischen Randbedingungen laufen. Abschließend werden für jeden Modus entweder die besten erfolgreichen Runs (über Seeds) animiert und/oder der C-Space mit Hindernisbelegung, Roadmap und Pfad geplottet; die Benchmarks werden dabei optional mit Roboterradius/Sicherheitsmarge aufgebaut.

In [ ]:
# Konfiguration
import math
import random
import numpy as np

MODE_ORDER = ["baseline_uniform", "mode1_seed_gauss", "mode2_seed_dist", "mode3_max_min", "mode4_start_goal_corr"]

base_planar = {
    "initialRoadmapSize": 2,
    "updateRoadmapSize":  3,
    "kNearest":           4,
    "maxIterations":      60,
}

MODE_PARAMS = {
    "baseline_uniform": {},
    "mode1_seed_gauss": dict(seedSigma=4.0, seedTries=5),
    "mode2_seed_dist":  dict(seedMaxStep=4.0, seedBeta=0.9, seedTries=5),
    "mode3_max_min":    dict(dispersionCandidates=5),
    "mode4_start_goal_corr": dict(corridorSigma=3.0, corridorAlongSigma=0.6, corridorTries=6),
}

def build_mode_configs(base_cfg, mode_params, mode_order):
    cfgs = {}
    for mode in mode_order:
        cfgs[mode] = dict(base_cfg, enhanceMode=mode, **mode_params.get(mode, {}))
    return cfgs

mode_configs = build_mode_configs(base_planar, MODE_PARAMS, MODE_ORDER)

def inject_overrides_for_benchmark(b, start_q, goal_q, configs_in, max_tries=20000):
    set_benchmark_start_goal(b, start=start_q, goal=goal_q, check=True)
    cc = b.collisionChecker
    start_list, goal_list = ensure_valid_start_goal(b, cc, max_tries=max_tries)

    start_ok = start_list[0]
    goal_ok  = goal_list[0]

    cfgs = {}
    for mode, cfg in configs_in.items():
        c = dict(cfg)
        c["start_override"] = start_ok
        c["goal_override"]  = goal_ok
        cfgs[mode] = c

    return cfgs, start_ok, goal_ok

def animate_best_runs_for_benchmark(
    b, start_q, goal_q,
    mode_configs, mode_order,
    seed_map,
    ws_limits=(-3, 3, -3, 3),
):
    """
    Animiert pro Mode den BESTEN erfolgreichen Run anhand seed_map[(benchmark_name, mode)] = seed.
    seed_map sollte aus extract_best_seed_map(df) stammen.
    """
    # Start/Goal einmal fix + validieren (und in configs injizieren)
    cfgs_b, start_ok, goal_ok = inject_overrides_for_benchmark(b, start_q, goal_q, mode_configs)

    print("\n--- Animations (BEST RUN per mode) for benchmark:", b.name, "---")
    print("Using start:", start_ok, " goal:", goal_ok)

    for mode in mode_order:
        key = (b.name, mode)
        if key not in seed_map:
            print(f"\nMode: {mode} -> no successful run in df (skip).")
            continue

        best_seed = int(seed_map[key])
        print(f"\nMode: {mode} -> animating best seed = {best_seed}")

        run_and_animate_planar(
            b=b,
            config=cfgs_b[mode],
            start_q=start_ok,
            goal_q=goal_ok,
            seed=best_seed,
            ws_limits=ws_limits
        )

import numpy as np
import random
from IPython.display import HTML, display
import task2b_planar_module as t2b

def run_and_animate_planar(b, config, start_q, goal_q, seed=None, ws_limits=(-3, 3, -3, 3)):
    # deterministische Seeds
    if seed is not None:
        random.seed(int(seed))
        np.random.seed(int(seed))

    # Start/Goal in Benchmark setzen (kollisionsfrei prüfen)
    t2b.set_benchmark_start_goal(b, start=start_q, goal=goal_q, check=True)

    # Planen
    planner = EnhancedLazyPRM(b.collisionChecker)
    path_nodes = planner.planPath([start_q], [goal_q], config)

    if not path_nodes or len(path_nodes) < 2:
        print("No valid path found -> skip animation.")
        return None

    # Pfad als Konfigurationen (theta1, theta2) aus Graph holen
    path_q = []
    for n in path_nodes:
        if n in planner.graph.nodes and "pos" in planner.graph.nodes[n]:
            path_q.append(list(planner.graph.nodes[n]["pos"]))

    # Animation (Arm)
    ani = t2b.animate_workspace_and_cspace(
        benchmark=b,
        planner=planner,
        path_q=path_q,
        q_limits=[(-np.pi, np.pi), (-np.pi, np.pi)],
        ws_limits=ws_limits,
        robot_kind="arm",
        show_cspace_obstacles=True,
        show_clearance=True,
        start_q=start_q,
        goal_q=goal_q,
    )

    display(HTML(ani.to_jshtml()))
    return ani

def animate_best_runs_for_benchmark_from_cfgs(
    b,
    cfgs_b,          # configs MIT start_override/goal_override (aus der Suite!)
    mode_order,
    seed_map,
    ws_limits=(-3, 3, -3, 3),
):
    # Start/Goal direkt aus cfgs_b lesen (identisch zur Suite)
    any_mode = mode_order[0]
    start_ok = cfgs_b[any_mode]["start_override"]
    goal_ok  = cfgs_b[any_mode]["goal_override"]

    print("\n--- Animations (BEST RUN per mode) for benchmark:", b.name, "---")
    print("Using start:", start_ok, " goal:", goal_ok)

    for mode in mode_order:
        key = (b.name, mode)
        if key not in seed_map:
            print(f"\nMode: {mode} -> no successful run in df (skip).")
            continue

        best_seed = int(seed_map[key])
        print(f"\nMode: {mode} -> animating best seed = {best_seed}")

        # RNG pro Mode deterministisch setzen (wichtig!)
        random.seed(best_seed)
        np.random.seed(best_seed)

        run_and_animate_planar(
            b=b,
            config=cfgs_b[mode],   # exakt die Suite-Config
            start_q=start_ok,
            goal_q=goal_ok,
            seed=best_seed,
            ws_limits=ws_limits
        )

# Hindernisse in C-Space anzeigen
def plot_cspace_per_mode(
    benchmark,
    cfgs_b,                 # z.B. cfgs_easy oder cfgs_pass (mit overrides!)
    mode_order,
    q_limits=[(-math.pi, math.pi), (-math.pi, math.pi)],
    n_occ=220,
):
    # Hindernisse nur einmal berechnen (schneller)
    occ = compute_cspace_occupancy(benchmark, q_limits, n=n_occ)

    # Start/Goal aus overrides nehmen (einheitlich)
    any_mode = mode_order[0]
    start_q = cfgs_b[any_mode]["start_override"]
    goal_q  = cfgs_b[any_mode]["goal_override"]

    for mode in mode_order:
        config = cfgs_b[mode]

        # Planner einmal laufen lassen
        planner = EnhancedLazyPRM(benchmark.collisionChecker)
        path_nodes = planner.planPath([start_q], [goal_q], config)

        ok = bool(path_nodes) and len(path_nodes) >= 2
        title = f"{benchmark.name} – {mode} | success={ok} | nodes={planner.graph.number_of_nodes()}"

        plot_cspace_with_roadmap(
            benchmark=benchmark,
            planner=planner,
            path_nodes=path_nodes if ok else None,
            q_limits=q_limits,
            occ=occ,
            title=title,
            start_q=start_q,
            goal_q=goal_q,
            draw_edges=True,
            max_edges=3000,
        )

# C-Space Plot Helpers
import numpy as np
import matplotlib.pyplot as plt
import math

def _q_in_collision(cc, q):
    if hasattr(cc, "pointInCollision"):
        return bool(cc.pointInCollision(q))
    if hasattr(cc, "configInCollision"):
        return bool(cc.configInCollision(q))
    raise AttributeError("CollisionChecker hat weder pointInCollision(q) noch configInCollision(q).")

def compute_cspace_occupancy(benchmark, q_limits, n=220):
    cc = benchmark.collisionChecker
    (q1min, q1max), (q2min, q2max) = q_limits
    q1 = np.linspace(q1min, q1max, n)
    q2 = np.linspace(q2min, q2max, n)

    occ = np.zeros((n, n), dtype=np.uint8)
    for i, a in enumerate(q1):
        for j, b in enumerate(q2):
            occ[j, i] = 1 if _q_in_collision(cc, [float(a), float(b)]) else 0
    return occ

def plot_cspace_with_roadmap(
    benchmark,
    planner,
    path_nodes=None,
    q_limits=[(-math.pi, math.pi), (-math.pi, math.pi)],
    occ=None,
    n=220,
    title="",
    start_q=None,
    goal_q=None,
    draw_edges=True,
    max_edges=3000,
):
    (q1min, q1max), (q2min, q2max) = q_limits

    # Hindernisse (Background)
    if occ is None:
        occ = compute_cspace_occupancy(benchmark, q_limits, n=n)

    fig, ax = plt.subplots(figsize=(6.8, 5.6))
    ax.imshow(
        occ,
        origin="lower",
        extent=[q1min, q1max, q2min, q2max],
        aspect="auto",
        interpolation="nearest",
        alpha=0.9
    )

    # Roadmap Overlay
    G = planner.graph
    nodes = list(G.nodes)
    if nodes:
        pts = np.array([G.nodes[n]["pos"] for n in nodes], dtype=float)
        ax.scatter(pts[:, 0], pts[:, 1], s=8, alpha=0.8, label="Roadmap nodes")

    if draw_edges:
        cnt = 0
        for u, v in G.edges:
            pu = G.nodes[u]["pos"]
            pv = G.nodes[v]["pos"]
            ax.plot([pu[0], pv[0]], [pu[1], pv[1]], linewidth=0.6, alpha=0.25)
            cnt += 1
            if cnt >= max_edges:
                break

    # Pfad Overlay
    if path_nodes:
        path_q = [G.nodes[p]["pos"] for p in path_nodes]
        pq = np.array(path_q, dtype=float)
        ax.plot(pq[:, 0], pq[:, 1], linewidth=2.0, alpha=0.95, label="Path")

    # Start / Goal
    if start_q is None:
        start_q = benchmark.startList[0]
    if goal_q is None:
        goal_q = benchmark.goalList[0]

    ax.scatter([start_q[0]], [start_q[1]], s=70, marker="o", label="start")
    ax.scatter([goal_q[0]], [goal_q[1]], s=70, marker="x", label="goal")

    ax.set_xlabel(r"$\theta_1$ [rad]")
    ax.set_ylabel(r"$\theta_2$ [rad]")
    ax.set_title(title or f"{benchmark.name} – C-Space with Roadmap")
    ax.legend(loc="upper right")
    plt.show()
    return fig, ax


In [ ]:
# 1) Modul importieren (Dateiname anpassen, falls anders)
import task2b_planar_module as t2b
from task2b_planar_module import set_benchmark_start_goal
from task2b_planar_module import ensure_valid_start_goal

# 2) Benchmarks bauen (Clearance = robot_radius + safety_margin)
#    -> robot_radius z.B. für "Robotershape berücksichtigen" > 0 setzen
bench_list = t2b.build_2b_benchmarks_from_task1style(
    robot_radius=0.0,     # z.B. 0.15 oder 0.20 wenn du Shape/Radius berücksichtigen willst
    safety_margin=0.0
)

# Rückgabe-Reihenfolge laut Funktion:
# [bP_easy, bP_passage, bA_easy, bA_passage]
b_point_easy, b_point_passage, b_planar_easy, b_planar_passage = bench_list

print("OK:", b_planar_easy.name, "|", b_planar_passage.name)

# 3) Start/Goal Variablen setzen (falls du sie weiter unten verwendest)
start_q_easy = b_planar_easy.startList[0]
goal_q_easy  = b_planar_easy.goalList[0]

start_q_pass = b_planar_passage.startList[0]
goal_q_pass  = b_planar_passage.goalList[0]

# Optional: sicherstellen, dass Start/Goal kollisionsfrei sind (und ggf. resampeln)
# (macht Sinn, wenn du Clearance erhöhst)
t2b.ensure_valid_start_goal(b_planar_easy, b_planar_easy.collisionChecker)
t2b.ensure_valid_start_goal(b_planar_passage, b_planar_passage.collisionChecker)

## Benchmark 1: Planar Easy

In [ ]:
# Plot & Animation
# --- Planar Easy ---
benchmarks = [b_planar_easy]

# 1) Für die Suite Start/Goal fixieren und Overrides injizieren
cfgs_easy, start_ok_easy, goal_ok_easy = inject_overrides_for_benchmark(
    b_planar_easy, start_q_easy, goal_q_easy, mode_configs
)

# 2) Suite laufen lassen (30 Runs pro Mode)
df_easy = run_suite(
    planner_class=EnhancedLazyPRM,
    benchmarks=benchmarks,
    configs=cfgs_easy,         # enthält bereits start_override/goal_override
    runs=30,
    base_seed=2025,
    progress_every=10,
    bench_mode_map=None
)

# 3) Summary-Plot
summary_easy = summarize(df_easy)
plot_benchmark_like_task1(summary_easy, b_planar_easy.name, mode_order=MODE_ORDER)

# 4) Best-Seed-Map aus df ziehen (pro Mode der beste erfolgreiche Run)
SEED_MAP_EASY = extract_best_seed_map(
    df_easy,
    prefer_seed_runs_for_mode12=True
)

# 5) Pro Mode den BESTEN erfolgreichen Run animieren
animate_best_runs_for_benchmark(
    b=b_planar_easy,
    start_q=start_ok_easy,
    goal_q=goal_ok_easy,
    mode_configs=mode_configs,
    mode_order=MODE_ORDER,
    seed_map=SEED_MAP_EASY,
    ws_limits=(-3, 3, -3, 3),
)

### Gesamtbewertung Planar_Easy

Der Benchmark Planar_Easy zeichnet sich durch einen gutartigen, zusammenhängenden Konfigurationsraum ohne enge Engpässe aus. Entsprechend erreichen alle Sampling-Modi eine hohe Erfolgsrate, sodass Unterschiede weniger in der Lösbarkeit als vielmehr in Effizienz und Struktur der erzeugten Roadmaps sichtbar werden. Die Planungszeit unterscheidet sich nur moderat zwischen den Modi. Die Baseline zeigt stabile, aber nicht minimale Laufzeiten. Lokal verdichtende Verfahren (Mode 1 und Mode 2) erhöhen die Planungszeit und deren Varianz leicht, ohne einen strukturellen Vorteil zu liefern. Mode 3 (Max–Min) erzielt die geringsten Planungszeiten, da die globale Abdeckung des Konfigurationsraums unnötige lokale Verdichtung vermeidet. Mode 4 (Start–Goal Corridor) liegt zeitlich im Bereich der Baseline, zeigt jedoch eine erhöhte Streuung.Ein ähnliches Bild zeigt sich bei den Kollisionschecks. Lokal seed-basierte Strategien verursachen den höchsten Prüfaufwand, während Mode 3 die geringste Anzahl an Kollisionschecks aufweist. Die Pfadlängen unterscheiden sich zwischen den Modi nur geringfügig und bleiben weitgehend unabhängig von der Sampling-Strategie. Die deutlichsten Unterschiede zeigen sich bei der Roadmapgröße. Während Baseline sowie Mode 1 und Mode 2 relativ große und stark streuende Roadmaps erzeugen, baut Mode 3 sehr kompakte und stabile Strukturen auf. Mode 4 liegt zwischen diesen Extremen. Insgesamt bestätigt Planar_Easy, dass bei einfacher Konfigurationsraumstruktur globale, sparsame Sampling-Strategien am effizientesten sind, während lokal verdichtende oder stark gerichtete Ansätze keinen wesentlichen Mehrwert bieten.


## Benchmark 2: Planar Passage

In [ ]:
# Plot & Animation
# --- Planar Passage ---
benchmarks = [b_planar_passage]

# 1) Suite (30 Runs) - Start/Goal einmal fixieren + Overrides injizieren
cfgs_pass, _, _ = inject_overrides_for_benchmark(
    b_planar_passage, start_q_pass, goal_q_pass, mode_configs
)

df_pass = run_suite(
    planner_class=EnhancedLazyPRM,
    benchmarks=[b_planar_passage],
    configs=cfgs_pass,      # enthält start_override/goal_override
    runs=30,
    base_seed=2025,
    progress_every=10,
    bench_mode_map=None
)

# 2) Summary Plot
summary_pass = summarize(df_pass)
plot_benchmark_like_task1(summary_pass, b_planar_passage.name, mode_order=MODE_ORDER)

# 3) Best-Seed-Map aus df_pass ziehen
SEED_MAP_PASS = extract_best_seed_map(df_pass, prefer_seed_runs_for_mode12=True)

# Plot Hindernisse im C-Space
#plot_cspace_per_mode(
 #   benchmark=b_planar_passage,
  #  cfgs_b=cfgs_pass,
   # mode_order=MODE_ORDER,
    #n_occ=220
#)

# 4) Animationen: pro Mode den besten Run (mit denselben overrides wie Suite)
animate_best_runs_for_benchmark_from_cfgs(
    b=b_planar_passage,
    cfgs_b=cfgs_pass,       # <<< entscheidend: NICHT neu injizieren
    mode_order=MODE_ORDER,
    seed_map=SEED_MAP_PASS,
    ws_limits=(-3, 3, -3, 3),
)


### Gesamtbewertung Planar_Easy

Der Benchmark Planar_Passage ist durch einen schmalen, stark eingeschränkten Durchgang im Konfigurationsraum gekennzeichnet und stellt damit deutlich höhere Anforderungen an die Strukturierung der Roadmap als Planar_Easy. Grundsätzlich sind alle Sampling-Modi in der Lage, gültige Pfade zu finden, Unterschiede zeigen sich jedoch klar in Effizienz, Pfadqualität und strukturellem Aufwand. Die Planungszeit und die Anzahl der Kollisionschecks unterscheiden sich deutlich zwischen den Modi. Uniformes Sampling verursacht den höchsten Aufwand, da große irrelevante Bereiche des Konfigurationsraums exploriert werden. Lokal seed-basierte Strategien reduzieren diesen Aufwand spürbar, bleiben jedoch anfällig für ungünstige Verdichtungen. Mode 4 (Start–Goal Corridor) erweist sich als besonders effizient, da die gezielte Einschränkung des Suchraums optimal zur Benchmarkstruktur passt. Mode 3 (Max–Min) zeigt dagegen erhöhte Kosten, da globale Exploration in diesem Szenario überwiegend ineffizient ist. Bei der Pfadlänge zeigen sich klare Vorteile für strukturierte Strategien. Während die Baseline längere und stärker variierende Pfade erzeugt, liefern Mode 2, Mode 3 und insbesondere Mode 4 deutlich direktere Trajektorien entlang des Durchgangs. Die Roadmapgröße bestätigt diesen Befund: Uniformes Sampling führt zu unnötig großen Roadmaps, während distanzbasierte und korridozentrische Strategien kompaktere Strukturen erzeugen. Die Erfolgsraten liegen für alle Modi auf einem hohen Niveau und unterscheiden sich nur moderat. Dies verdeutlicht, dass der Benchmark grundsätzlich lösbar ist, die Wahl des Sampling-Verfahrens jedoch maßgeblich bestimmt, wie effizient und strukturiert diese Lösung erreicht wird. Insgesamt zeigt Planar_Passage, dass bei stark eingeschränkten Konfigurationsräumen gezielte Start–Ziel-Fokussierung und kontrollierte lokale Strukturierung klar gegenüber globaler Exploration überlegen sind.

# Gesamtfazit der Arbeit

Ziel der Arbeit ist es, verschiedene Sampling- und Node-Enhancement-Strategien im Rahmen eines Lazy-PRM-Planers systematisch zu vergleichen und ihren Einfluss auf Lösbarkeit, Effizienz und Strukturqualität der erzeugten Roadmaps zu untersuchen. Die Evaluation erfolgt konsistent über mehrere Benchmarks hinweg anhand einheitlicher Metriken wie Erfolgsrate, Planungszeit, Kollisionschecks, Pfadlänge sowie Roadmapgröße und wurde zusätzlich durch qualitative Pfad- und C-Space-Visualisierungen unterstützt. Dadurch entsteht ein belastbares Gesamtbild darüber, unter welchen geometrischen und topologischen Bedingungen die einzelnen Modi ihre Stärken ausspielen und wann sie strukturell an Grenzen stoßen.

Über alle Experimente hinweg bestätigt sich, dass die Leistungsunterschiede der Sampling-Modi weniger durch „bessere“ oder „schlechtere“ Algorithmen im absoluten Sinn entstehen, sondern primär durch das Zusammenspiel aus 
- Benchmarkstruktur (Engpassbreite, Engpasslänge, Topologie, Irreführung),
- Roadmap-Strukturierung (global vs. lokal), und
- Modellannahmen (Punktroboter ohne Radius vs. Robotershape bzw. artikulierter Planarroboter).

**Zentrale Erkenntnisse zu den Sampling-Modi**

Baseline (uniform) erweist sich als robuster Referenzmodus in „gutartigen“ oder offenen Szenarien, in denen keine gezielte Strukturierung erforderlich ist. Gleichzeitig zeigt die Arbeit, dass eine geringe Planungszeit oder wenige Kollisionschecks bei der Baseline kein Qualitätsbeweis sind: In Engpass- und Irreführungsbenchmarks kann uniformes Sampling zwar schnell sein, aber rein zufallsgetrieben bleiben und dadurch strukturell unzuverlässig werden.

Mode 1 (seed_gauss) und Mode 2 (seed_dist) repräsentieren lokal verdichtende Strategien. Ihre Stärke liegt grundsätzlich darin, Konnektivität in bereits vielversprechenden Regionen zu erhöhen. In der Praxis zeigt die Evaluation jedoch, dass dieser Vorteil stark kontextabhängig ist: In offenen Räumen führen Seeds häufig zu Overbuilding ohne Mehrwert; in Engpasssituationen können sie zwar Pfadqualität verbessern, erzeugen aber meist deutlich höhere Kosten (Planungszeit/Kollisionschecks/Roadmapgröße). Besonders kritisch wird lokale Verdichtung in topologisch irreführenden Szenarien (U-Shape), wo sie systematisch im falschen Raum verdichtet und damit Aufwand produziert, ohne das eigentliche Problem (notwendiger Umweg) zuverlässig zu adressieren.

Mode 3 (Max–Min) fungiert als global explorierende Strategie mit struktureller Sparsamkeit. Die Ergebnisse zeigen zwei konsistente Charakteristika: erstens sehr kompakte Roadmaps (hohe strukturelle Effizienz), zweitens hohe Robustheit in Szenarien, in denen globale Exploration zwingend ist (Snail, teilweise stark eingeschränkte Situationen). Gleichzeitig steigt Varianz und Kosten, sobald die Problemstruktur eigentlich lokale oder gerichtete Abdeckung verlangt: In langen Passagen oder bei ineffizientem Abgleich zwischen schlecht abgedeckten Regionen und relevanten Regionen kann Max–Min unnötig in globalen Raum investieren. Mode 3 ist damit der beste Explorationsmodus, aber nicht automatisch der effizienteste Modus.

Mode 4 (Start–Goal Corridor) ist der stärkste Spezialist: Wo der direkte Start–Ziel-Zusammenhang geometrisch sinnvoll ist, liefert er sehr gute Effizienz und zugleich gute Pfadqualität. Wo jedoch ein Umweg strukturell notwendig oder die Umgebung topologisch irreführend ist (U-Shape, Snail), wird Mode 4 systematisch fehlgeleitet und kann vollständig scheitern.

**Einfluss der Robotershape im Punktroboter-Setting**

Die Gegenüberstellung „ohne Robotershape“ vs. „mit Robotershape“ zeigt, dass die Berücksichtigung eines Roboterradius nicht nur quantitative Effekte (mehr Kollisionen, längere Pfade) verursacht, sondern Benchmarks qualitativ in andere Problemklassen verschieben kann. Besonders deutlich wird dies bei Engpässen: Durch aufgeblasene Hindernisse schrumpft der freie Raum und damit die Trefferwahrscheinlichkeit gültiger Konfigurationen. In mehreren Fällen sinken Planungszeiten trotz höherer Schwierigkeit, was auf ein frühzeitiges Scheitern zurückzuführen ist. Gleichzeitig zeigen die Ergebnisse auch Gegenbeispiele: In U-Shape ändert sich fast nichts, weil die Robotershape die Topologie nicht fundamental verändert. Die Robotershape nimmt nur dann Einfluss, wenn sich die Engpassgeometrie oder Konnektivität im freien Raum tatsächlich verändert.

**Planarroboter und Bedeutung des C-Space**

Mit dem Wechsel zum 2-DoF-Planarroboter verschiebt sich das Problem von kartesischer Navigation hin zu Planung im Gelenkwinkelraum. Hindernisse werden im C-Space zu nichtlinearen, häufig verschachtelten Regionen und die Nähe zu Hindernissen ist im Samplingraum nicht mehr intuitiv erkennbar. Planar_Easy bestätigt, dass bei gutartiger C-Space-Struktur globale, sparsame Exploration (Mode 3) effizient ist und lokale Verdichtung primär Kosten verursacht. Planar_Passage zeigt dagegen, dass gerichtete oder lokal strukturierende Strategien in schmalen Konfigurationsraumkanälen deutliche Vorteile haben; hier ist Mode 4 besonders passend, während globale Exploration (Mode 3) ineffizient wird. 

**Zusammenführende Schlussfolgerung**

Es gibt keinen dominierenden Sampling-Modus über alle Szenarien hinweg. Stattdessen entsteht ein stabiler Trade-off zwischen globaler Exploration, lokaler Verdichtung und gerichteter Fokussierung:

- Offene, gutartige Räume: Baseline und Mode 3 sind effizient; lokale Seeds liefern kaum Nutzen.
- Schmale, lineare Passagen: Mode 4 und teilweise Mode 1 & 2 strukturieren den relevanten Raum am besten.
- Topologisch irreführende Umgebungen und verschachtelte Korridore: Mode 3 ist entscheidend; Mode 4 und seed-basierte Verfahren können strukturell scheitern.

Mit der Robotergeometrie steigt die Bedeutung korrekter Strukturierung; reine Knotenzahl ist kein Qualitätsindikator.